# Reviewer entry point: seeds 43 and 44

Use the same frozen release and working directory as Notebook 02. This notebook
writes its seed-43/44 runs separately. For the three-seed summary, place the
original seed-42 benchmark ZIP at the path named in the archive-restore cell
and keep all three runs on the recorded protocol. Set `RUN_TEST=True` in the
experiment controls only after validation choices are fixed. See Notebook 01
for data and environment setup.


# MahimandalSR review replication: seeds 43 and 44

Run from your own `25mcsa19` directory using your assigned GPU. This derives
the established isolated DGX workflow; it does not read any other student's
directory, perform global installations, or change the original results.

Existing dataset/source caches are reused under `./mahimandal160_25mcsa19`.
New outputs go to `outputs/mahimandal160_review_seeds_v1`, not the original
`mahimandal160_paper_benchmark_v1`. All train/validation/test images use full
160x160 LR -> 480x480 HR. Only reflectance16 is run. The seven published RGB
implementations are predeclared; the registry permits four-band runs only
for RDN, CAFRN and SRVGG. Fresh preflight results gate every model/modality.

The original seed 42 is not rerun. Combine it with seeds 43/44 only after
checking dataset/model/training protocol agreement. Do not choose a winning
seed or change epoch limits based on test. Keep RUN_TEST=False until the
validation-only choices are locked; set it True and rerun the test cell once.

Patience stays 3 to replicate the original protocol. Therefore SRVGG might
again stop before its GAN phase: that remains a pixel-only experiment, not
a new trained-RealESRGAN comparison. No source-disjoint or corrected-mask
training is implied. New seeds measure fit variability, not data independence.

This notebook was generated and statically checked locally, not GPU executed.


# MahimandalSR paper benchmark on DGX: 25MCSA19

This is the DGX conversion of
`Mahimandal_RGBNIR_160x160_Paper_Benchmark.ipynb`. It trains pinned x3
architectures on the locked 4,800-pair Mahimandal RGB+NIR release.

## Strict storage boundary

Launch this notebook from your own `25mcsa19` directory. The notebook reads
and writes task data only below the following relative folder:

`./mahimandal160_25mcsa19`

It does **not** inspect parent paths or sibling student directories. It does
not install a user/global kernelspec, use global pip, or write shared caches.
The only reads outside this relative workspace are system executables and
libraries required to run Python, CUDA and Git.

Before starting:

1. Ask the DGX administrator to expose exactly one allocated GPU to this
   notebook through `CUDA_VISIBLE_DEVICES`.
2. Put `kaggle.json` at `./kaggle.json`. After validation it is copied to
   `./mahimandal160_25mcsa19/secrets/kaggle/kaggle.json`. No directory search is
   performed.
3. Run model-training cells sequentially. Reruns resume from each model's
   checkpoint inside the relative workspace.

Test evaluation stays locked until validation decisions are final.

The primary protocol uses complete `160×160` Landsat inputs and aligned
`480×480` Sentinel-2 targets during training, validation and testing. No
random `64×64` training crop is used.


## 0. Establish the guarded runtime and local environment


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, urllib.request, zipfile

# All project locations are derived from the notebook launch directory.
# Start Jupyter from your own 25mcsa19 folder before running this cell.
SESSION_ROOT = Path(".").resolve()
WORKSPACE_ROOT = (Path(".") / 'mahimandal160_25mcsa19').resolve()
if WORKSPACE_ROOT.parent != SESSION_ROOT:
    raise RuntimeError(f"Unexpected relative workspace: {WORKSPACE_ROOT}")
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

def path_is_within(path, root):
    # Path-containment helper compatible with the DGX Python 3.8 runtime.
    candidate = Path(path).expanduser().resolve(strict=False)
    boundary = Path(root).expanduser().resolve(strict=False)
    try:
        candidate.relative_to(boundary)
    except ValueError:
        return False
    return True

def assert_owned(path, *, allow_missing=True):
    candidate = Path(path).expanduser().resolve(strict=False)
    if not path_is_within(candidate, WORKSPACE_ROOT):
        raise RuntimeError(
            f"Refusing access outside the relative benchmark workspace: {candidate}"
        )
    if not allow_missing and not candidate.exists():
        raise FileNotFoundError(candidate)
    return candidate

def assert_session_path(path, *, allow_missing=True):
    candidate = Path(path).expanduser().resolve(strict=False)
    if not path_is_within(candidate, SESSION_ROOT):
        raise RuntimeError(f"Refusing access above the notebook launch directory: {candidate}")
    if not allow_missing and not candidate.exists():
        raise FileNotFoundError(candidate)
    return candidate

DIRECTORIES = {
    "home": WORKSPACE_ROOT / "home",
    "cache": WORKSPACE_ROOT / ".cache",
    "tmp": WORKSPACE_ROOT / "tmp",
    "packages": WORKSPACE_ROOT / ".python-packages",
    "secrets": WORKSPACE_ROOT / "secrets" / "kaggle",
    "downloads": WORKSPACE_ROOT / "downloads" / "mahimandal160",
    "datasets": WORKSPACE_ROOT / "datasets" / "mahimandal160",
    "harness": WORKSPACE_ROOT / "benchmark_harness" / "mahimandal160",
    "sources": WORKSPACE_ROOT / "official_sources" / "mahimandal160",
    "outputs": WORKSPACE_ROOT / "outputs" / "mahimandal160_review_seeds_v1",
    "archives": WORKSPACE_ROOT / "archives" / "mahimandal160",
}
for path in DIRECTORIES.values():
    assert_owned(path)
    if path.exists() and path.is_symlink():
        raise RuntimeError(f"Owned runtime directory must not be a symlink: {path}")
    path.mkdir(parents=True, exist_ok=True)

HOME_ROOT = DIRECTORIES["home"]
CACHE_ROOT = DIRECTORIES["cache"]
TMP_ROOT = DIRECTORIES["tmp"]
HARNESS_DIR = DIRECTORIES["harness"]
SOURCES = DIRECTORIES["sources"]
OUTPUT = DIRECTORIES["outputs"]
ARCHIVE_ROOT = DIRECTORIES["archives"]
DOWNLOAD_ROOT = DIRECTORIES["downloads"]
DATASET_CONTAINER = DIRECTORIES["datasets"]
KAGGLE_DIR = DIRECTORIES["secrets"]
LOCAL_SITE = DIRECTORIES["packages"] / "site-packages"
LOCAL_SITE.mkdir(parents=True, exist_ok=True)
LOCAL_PYTHONPATH = os.pathsep.join((str(LOCAL_SITE), str(HARNESS_DIR)))

OWNED_ENVIRONMENT = {
    "HOME": str(HOME_ROOT),
    "XDG_CACHE_HOME": str(CACHE_ROOT),
    "PIP_CACHE_DIR": str(CACHE_ROOT / "pip"),
    "TORCH_HOME": str(CACHE_ROOT / "torch"),
    "HF_HOME": str(CACHE_ROOT / "huggingface"),
    "TRANSFORMERS_CACHE": str(CACHE_ROOT / "huggingface"),
    "MPLCONFIGDIR": str(CACHE_ROOT / "matplotlib"),
    "KAGGLE_CONFIG_DIR": str(KAGGLE_DIR),
    "KAGGLEHUB_CACHE": str(CACHE_ROOT / "kagglehub"),
    "PYTHONPYCACHEPREFIX": str(CACHE_ROOT / "pycache"),
    "TMPDIR": str(TMP_ROOT),
    "TMP": str(TMP_ROOT),
    "TEMP": str(TMP_ROOT),
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONUNBUFFERED": "1",
    "OMP_NUM_THREADS": "4",
    "MKL_NUM_THREADS": "4",
    "PYTHONPATH": LOCAL_PYTHONPATH,
}
os.environ.update(OWNED_ENVIRONMENT)
if str(LOCAL_SITE) not in sys.path:
    sys.path.insert(0, str(LOCAL_SITE))

def run(arguments, *, cwd=None, check=True, capture_output=False, text=True):
    command = [str(value) for value in arguments]
    cwd = WORKSPACE_ROOT if cwd is None else assert_owned(cwd, allow_missing=False)
    environment = os.environ.copy()
    environment.update(OWNED_ENVIRONMENT)
    environment["PYTHONPATH"] = LOCAL_PYTHONPATH
    print("+", " ".join(command), flush=True)
    return subprocess.run(
        command, cwd=cwd, env=environment, check=check,
        capture_output=capture_output, text=text,
    )

# Use the administrator-provided CUDA/PyTorch Python. Additional packages
# are installed only into LOCAL_SITE; python3-venv and sudo are not needed.
PYTHON = Path(sys.executable).resolve()
pip_probe = run(
    [PYTHON, "-m", "pip", "--version"], check=False, capture_output=True
)
if pip_probe.returncode != 0:
    bootstrap = assert_owned(DOWNLOAD_ROOT / "get-pip.py")
    if not bootstrap.is_file():
        bootstrap_url = (
            "https://bootstrap.pypa.io/pip/3.8/get-pip.py"
            if sys.version_info < (3, 9)
            else "https://bootstrap.pypa.io/get-pip.py"
        )
        print("Downloading local pip bootstrap:", bootstrap_url)
        urllib.request.urlretrieve(bootstrap_url, bootstrap)
    run([
        PYTHON, bootstrap, "--target", LOCAL_SITE,
        "--disable-pip-version-check", "--no-warn-script-location",
    ])
    pip_probe = run(
        [PYTHON, "-m", "pip", "--version"], check=False, capture_output=True
    )
    if pip_probe.returncode != 0:
        raise RuntimeError(
            "Local pip bootstrap failed; no system directories were modified."
        )

# A previous --target install may have pulled a second Torch through LPIPS.
# Preserve such files in a workspace-local quarantine and use DGX Torch.
local_torch_patterns = (
    "torch", "torch-*.dist-info", "torchvision", "torchvision-*.dist-info",
    "torchvision.libs", "torchaudio", "torchaudio-*.dist-info", "functorch",
    "torchgen",
)
local_torch_entries = sorted({
    entry for pattern in local_torch_patterns for entry in LOCAL_SITE.glob(pattern)
}, key=lambda entry: entry.name)
if local_torch_entries:
    quarantine = DIRECTORIES["packages"] / "quarantine" / (
        "duplicate_torch_" + time.strftime("%Y%m%d_%H%M%S")
    )
    quarantine.mkdir(parents=True, exist_ok=False)
    for entry in local_torch_entries:
        entry.rename(quarantine / entry.name)
    print("Quarantined workspace-local duplicate Torch packages:", quarantine)

if sys.version_info < (3, 9):
    # Python 3.8 is still common on shared DGX hosts. These are the newest
    # compatible scientific-package lines; newer releases require 3.9+.
    REQUIREMENTS = [
        "rasterio>=1.3,<1.4", "numpy>=1.24,<1.25", "pandas>=2.0,<2.1",
        "matplotlib>=3.7,<3.8", "scipy>=1.10,<1.11", "einops>=0.7",
        "pyyaml>=6", "fvcore>=0.1.5", "Pillow>=10,<11", "kaggle>=1.7",
        "ipykernel>=6.29,<7", "protobuf==3.20.3",
    ]
else:
    REQUIREMENTS = [
        "rasterio>=1.3", "numpy>=1.26", "pandas>=2", "matplotlib>=3.8",
        "scipy>=1.11", "einops>=0.7", "pyyaml>=6", "fvcore>=0.1.5",
        "Pillow>=10", "kaggle>=1.7", "ipykernel>=6.29",
        "protobuf==3.20.3",
    ]
NO_DEPENDENCY_REQUIREMENTS = ["lpips>=0.1.4"]
requirement_key = [
    sys.version.split()[0], *REQUIREMENTS, "--no-deps",
    *NO_DEPENDENCY_REQUIREMENTS,
]
requirement_id = hashlib.sha256("\n".join(requirement_key).encode()).hexdigest()
install_receipt = DIRECTORIES["packages"] / "mahimandal160-requirements.json"
receipt = json.loads(install_receipt.read_text()) if install_receipt.is_file() else {}
if receipt.get("sha256") != requirement_id:
    run([
        PYTHON, "-m", "pip", "install", "--upgrade", "--target", LOCAL_SITE,
        "--disable-pip-version-check", "setuptools", "wheel", *REQUIREMENTS,
    ])
    run([
        PYTHON, "-m", "pip", "install", "--upgrade", "--target", LOCAL_SITE,
        "--disable-pip-version-check", "--no-deps",
        *NO_DEPENDENCY_REQUIREMENTS,
    ])
    install_receipt.write_text(
        json.dumps({
            "sha256": requirement_id,
            "python": sys.version.split()[0],
            "target": str(LOCAL_SITE),
            "requirements": REQUIREMENTS,
            "no_dependency_requirements": NO_DEPENDENCY_REQUIREMENTS,
        }, indent=2),
        encoding="utf-8",
    )

print("Notebook launch directory:", SESSION_ROOT)
print("Relative benchmark workspace:", WORKSPACE_ROOT)
print("Benchmark Python:", PYTHON)
print("Workspace-only package directory:", LOCAL_SITE)
print("All mutable paths passed the ownership guard:")
for name, path in DIRECTORIES.items():
    print(f"  {name:10s} {path}")


## 1. Verify the administrator-assigned GPU


In [ ]:
# Do not choose a physical GPU here: preserve scheduler/cgroup isolation.
visible = os.environ.get("CUDA_VISIBLE_DEVICES", "").strip()
import torch
torch_path = Path(torch.__file__).resolve()
if path_is_within(torch_path, LOCAL_SITE):
    raise RuntimeError(
        f"Refusing workspace-local duplicate PyTorch: {{torch_path}}. "
        "Restart the kernel and rerun the setup cell so DGX PyTorch is used."
    )
gpu_count = torch.cuda.device_count()
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("PyTorch location:", torch_path)
print("CUDA_VISIBLE_DEVICES:", visible or "<unset>")
print("PyTorch-visible GPU count:", gpu_count)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable to this notebook kernel.")
if gpu_count != 1:
    raise RuntimeError(
        f"Expected exactly one scheduler/admin-isolated GPU, but PyTorch sees "
        f"{gpu_count}. Ask the DGX administrator to launch this notebook with "
        "one GPU; the benchmark will not select an arbitrary shared device."
    )
if not visible:
    print(
        "CUDA_VISIBLE_DEVICES is unset, but exactly one GPU is visible through "
        "PyTorch; accepting the scheduler/cgroup-isolated device."
    )
properties = torch.cuda.get_device_properties(0)
print("Allocated GPU:", properties.name)
print("VRAM GiB:", round(properties.total_memory / 2**30, 2))
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True


## 2. Authenticate with `kaggle.json` and fetch the reflectance16 release

The Kaggle username in `kaggle.json` is used to form
`<username>/mahimandal-sr-reflectance16`. Set `KAGGLE_DATASET_OVERRIDE` only
when the release belongs to a different Kaggle owner. No credential search is
performed outside the two documented paths below.


In [ ]:
from getpass import getpass

KAGGLE_DATASET_OVERRIDE = "twilight2002/mahimandal-sr-reflectance16"  # Example: 'owner/mahimandal-sr-reflectance16'
credential_candidates = (
    KAGGLE_DIR / "kaggle.json",
    SESSION_ROOT / "kaggle.json",
)
found_credentials = [path for path in credential_candidates if path.is_file()]
if not found_credentials:
    raise FileNotFoundError(
        "Place kaggle.json at one of these owned paths only: "
        + ", ".join(map(str, credential_candidates))
    )

credential_source = assert_session_path(found_credentials[0], allow_missing=False)
credentials = json.loads(credential_source.read_text(encoding="utf-8"))
if not credentials.get("username") or not credentials.get("key"):
    raise ValueError("kaggle.json must contain nonempty username and key fields")

KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"
if credential_source != KAGGLE_JSON:
    shutil.copy2(credential_source, KAGGLE_JSON)
KAGGLE_JSON.chmod(0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_DIR)

KAGGLE_DATASET = (
    KAGGLE_DATASET_OVERRIDE
    or f"{credentials['username']}/mahimandal-sr-reflectance16"
)
DATASET_SLUG = KAGGLE_DATASET.split("/", 1)[-1]
OUTER_ARCHIVE = assert_owned(DOWNLOAD_ROOT / f"{DATASET_SLUG}.zip")
OUTER_STAGE = assert_owned(DOWNLOAD_ROOT / "outer_extracted")

def archive_valid(path):
    if not Path(path).is_file():
        return False
    try:
        with zipfile.ZipFile(path) as archive:
            # Reading the central directory detects interrupted downloads.
            # CRC is checked while extracting, avoiding a second full pass.
            return bool(archive.infolist())
    except zipfile.BadZipFile:
        return False

def safe_extract(archive_path, destination):
    archive_path = assert_owned(archive_path, allow_missing=False)
    destination = assert_owned(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve(strict=False)
            if not path_is_within(target, destination):
                raise RuntimeError(f"ZIP member escapes destination: {member.filename}")
        archive.extractall(destination)

def dataset_candidates():
    candidates = []
    for protocol_path in DATASET_CONTAINER.rglob("protocol.json"):
        root = protocol_path.parent
        try:
            protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        manifest = root / "paper_release_manifest.csv"
        if (protocol.get("target_pairs") == 4800 and manifest.is_file()
                and (root / "train" / "LR").is_dir()
                and (root / "train" / "HR").is_dir()
                and (root / "val" / "LR").is_dir()
                and (root / "test" / "LR").is_dir()):
            candidates.append(root.resolve())
    return sorted(set(candidates))

candidates = dataset_candidates()
if not candidates:
    if OUTER_ARCHIVE.exists() and not archive_valid(OUTER_ARCHIVE):
        quarantine = OUTER_ARCHIVE.with_name(
            OUTER_ARCHIVE.stem + f".corrupt-{int(time.time())}.zip"
        )
        OUTER_ARCHIVE.replace(quarantine)
        print("Preserved corrupt download as:", quarantine)

    if not archive_valid(OUTER_ARCHIVE):
        run([
        PYTHON, "-c", "from kaggle.cli import main; main()",
        "datasets", "download",
            "-d", KAGGLE_DATASET, "-p", DOWNLOAD_ROOT,
        ])
    if not archive_valid(OUTER_ARCHIVE):
        raise RuntimeError(f"Kaggle download is incomplete: {OUTER_ARCHIVE}")

    extraction_marker = OUTER_STAGE / ".complete.json"
    if not extraction_marker.is_file():
        safe_extract(OUTER_ARCHIVE, OUTER_STAGE)
        extraction_marker.write_text(
            json.dumps({"dataset": KAGGLE_DATASET, "archive": str(OUTER_ARCHIVE)}, indent=2),
            encoding="utf-8",
        )

    nested_archives = sorted(OUTER_STAGE.rglob("*.zip"))
    if len(nested_archives) == 1:
        nested_marker = DATASET_CONTAINER / ".nested_extract_complete.json"
        if not nested_marker.is_file():
            safe_extract(nested_archives[0], DATASET_CONTAINER)
            nested_marker.write_text(
                json.dumps({"archive": str(nested_archives[0])}, indent=2),
                encoding="utf-8",
            )
    elif len(nested_archives) > 1:
        raise RuntimeError(f"Expected at most one packaged release ZIP: {nested_archives}")
    else:
        # Some Kaggle versions expose release contents directly.
        for item in OUTER_STAGE.iterdir():
            destination = DATASET_CONTAINER / item.name
            if not destination.exists() and item.name != ".complete.json":
                shutil.move(str(item), destination)

    candidates = dataset_candidates()

if len(candidates) != 1:
    raise RuntimeError(
        "Expected exactly one 4,800-pair reflectance16 release inside the "
        f"owned dataset folder; found: {candidates}"
    )
DATASET_ROOT = assert_owned(candidates[0], allow_missing=False)
print("Authenticated Kaggle user:", credentials["username"])
print("Dataset reference:", KAGGLE_DATASET)
print("Dataset root:", DATASET_ROOT)
print("Credential key was not printed.")


## 3. Materialize the immutable benchmark harness inside 25MCSA19


In [ ]:
# Snapshot generated from this repository revision.
HARNESS = {'registry.json': '{\n  "CAFRN": {"repo": "lxy0922/CAFRN", "revision": "fef57e89c46127ddd0ddaaee4599ec39f207e20f", "source_dir": "CAFRN", "entry": "cafrn.py", "provenance": "author implementation", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "extended"},\n  "DCM": {"repo": "Shaosifan/HSENet", "revision": "29e71e33a2d6cb56013886c7f621e793602507f7", "source_dir": "HSENet", "entry": "codes/model/dcm.py", "provenance": "comparison implementation distributed with HSENet; not verified as original DCM author code", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "FENet": {"repo": "wangzheyuan-666/FeNet", "revision": "165954b920c085b07762493193140411e8c4d08a", "source_dir": "FENet", "entry": "model/fenet.py", "provenance": "author Feature Enhancement Network implementation; acronym identity must be checked against the cited paper", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "HAUNet": {"repo": "likakakaka/HAUNet_RSISR", "revision": "53f0b0dec3ecde7e22c328eaf3a73bd8982eaf30", "source_dir": "HAUNet", "entry": "codes/model/haunet.py", "provenance": "author make_model configuration", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "HSENet": {"repo": "Shaosifan/HSENet", "revision": "29e71e33a2d6cb56013886c7f621e793602507f7", "source_dir": "HSENet", "entry": "codes/model/hsenet.py", "provenance": "author implementation", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "MHAN": {"repo": "ZhangDY827/MHAN", "revision": "e5864b078c76817e3bc6eb5af5442ddaeca2b3a7", "source_dir": "MHAN", "entry": "models/model.py", "provenance": "author implementation with published scale=3 interface", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "SAN": {"repo": "daitao/SAN", "revision": "7c1e61f0d17b58e3400a14e631d9185dde1881b6", "source_dir": "SAN", "entry": "TrainCode/model/san.py", "provenance": "author implementation", "range_multiplier": 255, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "EDSR": {"repo": "sanghyun-son/EDSR-PyTorch", "revision": "8dba5581a7502b92de9641eb431130d6c8ca5d7f", "source_dir": "EDSR", "entry": "src/model/edsr.py", "provenance": "PyTorch implementation maintained by the EDSR authors; original EDSR configuration", "range_multiplier": 255, "range_offset": 0, "modalities": ["rgb"], "suite": "core"},\n  "RCAN": {"repo": "yulunzhang/RCAN", "revision": "f722e5d0e1eff45148bfaefd9bc85c6f0f26ce02", "source_dir": "RCAN", "entry": "RCAN_TrainCode/code/model/rcan.py", "provenance": "author implementation", "range_multiplier": 255, "range_offset": 0, "modalities": ["rgb"], "suite": "core"},\n  "RDN": {"repo": "sanghyun-son/EDSR-PyTorch", "revision": "8dba5581a7502b92de9641eb431130d6c8ca5d7f", "source_dir": "EDSR", "entry": "src/model/rdn.py", "provenance": "PyTorch port linked by the original RDN authors; RDN-B", "range_multiplier": 255, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "core"},\n  "SwinIR": {"repo": "JingyunLiang/SwinIR", "revision": "6545850fbf8df298df73d81f3e8cba638787c8bd", "source_dir": "SwinIR", "entry": "models/network_swinir.py", "provenance": "author implementation; classical SR large model with native x3 PixelShuffle", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "core"},\n  "HAT": {"repo": "XPixelGroup/HAT", "revision": "1638a9a822581657811867bf670717f8371fc3e5", "source_dir": "HAT", "entry": "hat/archs/hat_arch.py", "provenance": "author HAT-S implementation with native x3 PixelShuffle", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "core"},\n  "OmniSR": {"repo": "Francis0625/Omni-SR", "revision": "cd9901e3e9055b44ec22e6b2c408a08d31ea3dd4", "source_dir": "OmniSR", "entry": "components/OmniSR.py", "provenance": "author X3 DIV2K configuration", "range_multiplier": 2, "range_offset": -1, "modalities": ["rgb", "rgbnir"], "suite": "core"},\n  "RealESRGAN_SRVGG_X3": {"repo": "xinntao/Real-ESRGAN", "revision": "a4abfb2979a7bbff3f69f58f58ae324608821e27", "source_dir": "RealESRGAN", "entry": "realesrgan/archs/srvgg_arch.py", "provenance": "official SRVGGNetCompact and U-Net discriminator, configured for x3 and trained on paired data; an explicit adaptation, not an official Real-ESRGAN x3 checkpoint", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "perceptual", "adaptation": "native arbitrary-scale constructor set to 3; common paired perceptual-GAN training replaces Real-ESRGAN blind degradation"}\n}\n', 'adapters.py': '"""Pinned upstream x3 model adapters with explicit modality capability checks."""\nfrom __future__ import annotations\n\nimport contextlib\nimport importlib.util\nimport io\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport types\nfrom types import SimpleNamespace\n\nimport torch\nfrom torch import nn\nimport yaml\n\n\nREGISTRY = json.loads(Path(__file__).with_name("registry.json").read_text(encoding="utf-8"))\n\n\nclass _Registry:\n    def register(self, obj=None, **_):\n        def decorator(value):\n            return value\n        return decorator(obj) if obj is not None else decorator\n\n\ndef _install_basicsr_stubs() -> None:\n    """Provide only the two BasicSR interfaces used by pinned architecture files."""\n    if "basicsr.utils.registry" in sys.modules:\n        return\n    basicsr = types.ModuleType("basicsr")\n    basicsr.__path__ = []\n    utils = types.ModuleType("basicsr.utils")\n    utils.__path__ = []\n    registry = types.ModuleType("basicsr.utils.registry")\n    registry.ARCH_REGISTRY = _Registry()\n    archs = types.ModuleType("basicsr.archs")\n    archs.__path__ = []\n    arch_util = types.ModuleType("basicsr.archs.arch_util")\n    arch_util.to_2tuple = lambda value: value if isinstance(value, tuple) else (value, value)\n    arch_util.trunc_normal_ = torch.nn.init.trunc_normal_\n    sys.modules.update({"basicsr": basicsr, "basicsr.utils": utils,\n                        "basicsr.utils.registry": registry, "basicsr.archs": archs,\n                        "basicsr.archs.arch_util": arch_util})\n\n\ndef git(root: str | Path, *arguments: str) -> str:\n    return subprocess.check_output(["git", "-C", str(root), *arguments], text=True).strip()\n\n\ndef verify_source(root: str | Path, spec: dict) -> None:\n    root = Path(root)\n    if git(root, "rev-parse", "HEAD") != spec["revision"]:\n        raise ValueError(f"Wrong source revision: {root}; use a separate sources directory")\n    if git(root, "status", "--porcelain", "--untracked-files=no"):\n        raise ValueError(f"Tracked upstream files were modified: {root}")\n\n\ndef fetch(sources: str | Path) -> None:\n    sources = Path(sources)\n    sources.mkdir(parents=True, exist_ok=True)\n    unique = {spec["source_dir"]: spec for spec in REGISTRY.values()}\n    for spec in unique.values():\n        root = sources / spec["source_dir"]\n        if not root.exists():\n            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",\n                            f"https://github.com/{spec[\'repo\']}.git", str(root)], check=True)\n            subprocess.run(["git", "-C", str(root), "sparse-checkout", "set", "--no-cone",\n                            "*.py", "*.yaml", "*.yml", "*.md", "*LICENSE*"], check=True)\n            subprocess.run(["git", "-C", str(root), "checkout", spec["revision"]], check=True)\n        verify_source(root, spec)\n        print(spec["source_dir"], spec["revision"], flush=True)\n\n\ndef _load_file(name: str, path: Path):\n    module_spec = importlib.util.spec_from_file_location(name, path)\n    if module_spec is None or module_spec.loader is None:\n        raise ImportError(f"Cannot import {name} from {path}")\n    module = importlib.util.module_from_spec(module_spec)\n    sys.modules[name] = module\n    module_spec.loader.exec_module(module)\n    return module\n\n\ndef _clear_upstream_modules() -> None:\n    for key in list(sys.modules):\n        if key.split(".")[0] in {"model", "models", "ops", "blocks", "components"}:\n            del sys.modules[key]\n\n\ndef _classic_model(name: str, module, channels: int):\n    args = dict(scale=[3], n_colors=channels, n_feats=64, rgb_range=255,\n                res_scale=1, n_resgroups=10, n_resblocks=20, reduction=16,\n                n_basic_modules=10, G0=64, RDNkSize=3, RDNconfig="B",\n                no_upsampling=False)\n    if name == "EDSR":\n        args.update(n_feats=256, n_resblocks=32, res_scale=0.1)\n    if name == "SAN":\n        args.update(n_resgroups=20, n_resblocks=10)\n    return module.make_model(SimpleNamespace(**args)), args\n\n\ndef load(name: str, sources: str | Path, modality: str):\n    if modality not in {"rgb", "rgbnir"}:\n        raise ValueError(modality)\n    channels = 3 if modality == "rgb" else 4\n    if name == "bicubic":\n        return None, {"name": "bicubic", "scale": 3, "input_channels": channels,\n                      "output_channels": channels, "parameters": 0,\n                      "provenance": "PyTorch bicubic interpolation; align_corners=False",\n                      "suite": "baseline", "range_multiplier": 1, "range_offset": 0}\n    if name not in REGISTRY:\n        raise KeyError(name)\n    spec = dict(REGISTRY[name])\n    if modality not in spec["modalities"]:\n        raise ValueError(f"{name} does not have a verified {modality} construction; supported={spec[\'modalities\']}")\n    root = (Path(sources) / spec["source_dir"]).resolve()\n    verify_source(root, spec)\n    entry = root / spec["entry"]\n    _clear_upstream_modules()\n    if name in {"DCM", "HSENet", "HAUNet", "RCAN", "SAN", "RDN", "EDSR"}:\n        import_root = entry.parent.parent\n    else:\n        import_root = root\n    sys.path.insert(0, str(import_root))\n    try:\n        with contextlib.redirect_stdout(io.StringIO()):\n            if name == "HAT":\n                _install_basicsr_stubs()\n            if name == "RealESRGAN_SRVGG_X3":\n                _install_basicsr_stubs()\n            module = _load_file("upstream_" + name, entry)\n            if name == "CAFRN":\n                kwargs = dict(in_channels=channels, out_channels=channels, num_features=64,\n                              num_steps=7, reduction=16, upscale_factor=3,\n                              act_type="prelu", norm_type=None)\n                model = module.CAFRN(**kwargs)\n            elif name == "FENet":\n                kwargs = dict(upscale_factor=3)\n                model = module.FENet(**kwargs)\n            elif name == "MHAN":\n                kwargs = dict(in_channels=channels, num_features=64, scale=3)\n                model = module.Net(**kwargs)\n            elif name == "OmniSR":\n                config = yaml.safe_load((root / "train_yamls/train_OmniSR_X3_DIV2K.yaml").read_text())\n                kwargs = dict(num_in_ch=channels, num_out_ch=channels,\n                              num_feat=config["feature_num"], **config["module_params"])\n                model = module.OmniSR(**kwargs)\n            elif name == "SwinIR":\n                kwargs = dict(img_size=64, patch_size=1, in_chans=channels, embed_dim=180,\n                              depths=(6, 6, 6, 6, 6, 6), num_heads=(6, 6, 6, 6, 6, 6),\n                              window_size=8, mlp_ratio=2, upscale=3, img_range=1.0,\n                              upsampler="pixelshuffle", resi_connection="1conv",\n                              use_checkpoint=False)\n                model = module.SwinIR(**kwargs)\n            elif name == "HAT":\n                kwargs = dict(img_size=64, in_chans=channels, embed_dim=144,\n                              depths=(6, 6, 6, 6, 6, 6), num_heads=(6, 6, 6, 6, 6, 6),\n                              window_size=16, compress_ratio=24, squeeze_factor=24,\n                              conv_scale=0.01, overlap_ratio=0.5, mlp_ratio=2,\n                              upscale=3, img_range=1.0, upsampler="pixelshuffle",\n                              resi_connection="1conv", use_checkpoint=False)\n                model = module.HAT(**kwargs)\n            elif name == "RealESRGAN_SRVGG_X3":\n                kwargs = dict(num_in_ch=channels, num_out_ch=channels, num_feat=64,\n                              num_conv=32, upscale=3, act_type="prelu")\n                model = module.SRVGGNetCompact(**kwargs)\n            else:\n                model, kwargs = _classic_model(name, module, channels)\n    finally:\n        if str(import_root) in sys.path:\n            sys.path.remove(str(import_root))\n    spec.update(name=name, constructor=kwargs, input_channels=channels,\n                output_channels=channels, scale=3,\n                parameters=sum(parameter.numel() for parameter in model.parameters()),\n                trainable_parameters=sum(parameter.numel() for parameter in model.parameters()\n                                         if parameter.requires_grad))\n    return model, spec\n\n\ndef load_discriminator(sources: str | Path) -> nn.Module:\n    spec = REGISTRY["RealESRGAN_SRVGG_X3"]\n    root = (Path(sources) / spec["source_dir"]).resolve()\n    verify_source(root, spec)\n    _install_basicsr_stubs()\n    module = _load_file("upstream_realesrgan_discriminator",\n                        root / "realesrgan/archs/discriminator_arch.py")\n    # The adversarial signal is deliberately RGB-only in RGB+NIR experiments.\n    return module.UNetDiscriminatorSN(num_in_ch=3, num_feat=64, skip_connection=True)\n\n\ndef predict(model: nn.Module | None, spec: dict, lr: torch.Tensor) -> torch.Tensor:\n    if model is None:\n        result = torch.nn.functional.interpolate(lr, scale_factor=3, mode="bicubic",\n                                                 align_corners=False)\n    else:\n        multiplier = float(spec["range_multiplier"])\n        offset = float(spec["range_offset"])\n        result = model(lr * multiplier + offset)\n        if not isinstance(result, torch.Tensor):\n            raise TypeError("Unexpected upstream output; auxiliary outputs are not silently selected")\n        result = (result - offset) / multiplier\n    expected = (lr.shape[0], spec["output_channels"], lr.shape[-2] * 3, lr.shape[-1] * 3)\n    if tuple(result.shape) != expected:\n        raise ValueError(f"Model returned {tuple(result.shape)}, expected {expected}")\n    return result\n', 'data.py': '"""Locked Mahimandal 160->480 RGB/RGB+NIR dataset reader."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nimport rasterio\nfrom rasterio.enums import ColorInterp\nimport torch\nfrom torch.utils.data import Dataset\n\n\nSCALE = 3\nLR_SIZE = 160\nHR_SIZE = 480\nBAND_ORDER = ("red", "green", "blue", "nir")\n\n\ndef digest(path: str | Path) -> str:\n    value = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            value.update(chunk)\n    return value.hexdigest()\n\n\ndef resolve(root: str | Path, path: str | Path) -> Path:\n    root = Path(root).resolve()\n    candidate = Path(path)\n    candidate = candidate if candidate.is_absolute() else root / candidate\n    candidate = candidate.resolve()\n    try:\n        candidate.relative_to(root)\n    except ValueError:\n        raise ValueError(f"Path escapes dataset root: {path}")\n    return candidate\n\n\ndef find_release_manifest(root: str | Path) -> Path:\n    root = Path(root)\n    candidates = (\n        root / "paper_reports" / "paper_release_manifest.csv",\n        root / "paper_release_manifest.csv",\n    )\n    found = [path for path in candidates if path.is_file()]\n    if len(found) != 1:\n        raise FileNotFoundError(f"Expected one paper_release_manifest.csv under {root}; found {found}")\n    return found[0]\n\n\ndef read_rows(root: str | Path) -> list[dict]:\n    root = Path(root)\n    with find_release_manifest(root).open(newline="", encoding="utf-8-sig") as stream:\n        rows = list(csv.DictReader(stream))\n    required = {"pair_id", "split", "landscape_class", "spatial_block_id"}\n    missing = required.difference(rows[0] if rows else {})\n    if missing:\n        raise ValueError(f"Release manifest lacks required columns: {sorted(missing)}")\n    for row in rows:\n        pair_id = row["pair_id"]\n        row["lr_path"] = row.get("lr_path") or f"landsat/{pair_id}.tif"\n        row["hr_path"] = row.get("hr_path") or f"sentinel/{pair_id}.tif"\n    return rows\n\n\ndef _band_indices(dataset: rasterio.io.DatasetReader, modality: str) -> list[int]:\n    names = [str(name or "").strip().casefold() for name in dataset.descriptions]\n    aliases = {"red": {"red", "b4", "sr_b4"}, "green": {"green", "b3", "sr_b3"},\n               "blue": {"blue", "b2", "sr_b2"}, "nir": {"nir", "b8", "sr_b5", "b5"}}\n    requested = BAND_ORDER[:3] if modality == "rgb" else BAND_ORDER\n    indices = []\n    for band in requested:\n        matches = [index + 1 for index, value in enumerate(names) if value in aliases[band]]\n        if len(matches) != 1:\n            raise ValueError(f"Cannot resolve one {band} band from descriptions {dataset.descriptions}")\n        indices.append(matches[0])\n    return indices\n\n\ndef _reflectance(array: np.ndarray, source_dtype: str, radiometry: str) -> np.ndarray:\n    array = array.astype(np.float32, copy=False)\n    is_integer = np.issubdtype(np.dtype(source_dtype), np.integer)\n    if radiometry == "normalized_float32" and is_integer:\n        raise ValueError("normalized_float32 requires floating-point TIFF storage")\n    if is_integer:\n        array = array / 10000.0\n    elif np.nanmax(array) > 1.5:\n        raise ValueError("Floating TIFF exceeds reflectance range; scaling is ambiguous")\n    array = np.clip(array, 0.0, 1.0)\n    if radiometry == "quantized8":\n        array = np.rint(array * 255.0) / 255.0\n    elif radiometry not in {"reflectance16", "normalized_float32"}:\n        raise ValueError(f"Unknown radiometry protocol: {radiometry}")\n    return array\n\n\nclass Pairs(Dataset):\n    def __init__(self, root: str | Path, split: str, modality: str = "rgb",\n                 radiometry: str = "reflectance16", augment: bool = False,\n                 lr_crop: int | None = None):\n        if modality not in {"rgb", "rgbnir"}:\n            raise ValueError("modality must be rgb or rgbnir")\n        if lr_crop is not None and not 1 <= lr_crop <= LR_SIZE:\n            raise ValueError(f"lr_crop must be in [1,{LR_SIZE}]")\n        self.root = Path(root)\n        self.modality = modality\n        self.radiometry = radiometry\n        self.augment = augment\n        self.lr_crop = lr_crop\n        self.rows = sorted((row for row in read_rows(root) if row["split"] == split),\n                           key=lambda row: row["pair_id"])\n        if not self.rows:\n            raise ValueError(f"Empty split {split!r}")\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, index: int):\n        row = self.rows[index]\n        with rasterio.open(resolve(self.root, row["lr_path"])) as source:\n            lr = _reflectance(source.read(_band_indices(source, self.modality)), source.dtypes[0], self.radiometry)\n            lr_mask = np.all(source.read_masks() > 0, axis=0)\n        with rasterio.open(resolve(self.root, row["hr_path"])) as source:\n            hr = _reflectance(source.read(_band_indices(source, self.modality)), source.dtypes[0], self.radiometry)\n            hr_mask = np.all(source.read_masks() > 0, axis=0)\n        channels = 3 if self.modality == "rgb" else 4\n        if lr.shape != (channels, LR_SIZE, LR_SIZE) or hr.shape != (channels, HR_SIZE, HR_SIZE):\n            raise ValueError(f"Unexpected geometry for {row[\'pair_id\']}: {lr.shape}, {hr.shape}")\n        joint_mask = hr_mask & np.repeat(np.repeat(lr_mask, SCALE, axis=0), SCALE, axis=1)\n        if not joint_mask.any() or not np.isfinite(lr).all() or not np.isfinite(hr).all():\n            raise ValueError(f"No finite jointly valid pixels for {row[\'pair_id\']}")\n        if self.lr_crop and self.lr_crop < LR_SIZE:\n            top = int(torch.randint(0, LR_SIZE - self.lr_crop + 1, ()).item())\n            left = int(torch.randint(0, LR_SIZE - self.lr_crop + 1, ()).item())\n            bottom, right = top + self.lr_crop, left + self.lr_crop\n            lr = lr[:, top:bottom, left:right]\n            hr = hr[:, top * SCALE:bottom * SCALE, left * SCALE:right * SCALE]\n            joint_mask = joint_mask[top * SCALE:bottom * SCALE, left * SCALE:right * SCALE]\n        arrays = [lr, hr, joint_mask[None].astype(np.float32)]\n        if self.augment:\n            rotations = int(torch.randint(4, ()).item())\n            arrays = [np.rot90(array, rotations, axes=(-2, -1)) for array in arrays]\n            if torch.rand(()) < 0.5:\n                arrays = [array[..., ::-1] for array in arrays]\n        return (*[torch.from_numpy(array.copy()) for array in arrays], index)\n\n\ndef audit(root: str | Path) -> dict:\n    root = Path(root)\n    rows = read_rows(root)\n    counts = Counter(row["split"] for row in rows)\n    if set(counts) != {"train", "val", "test"} or min(counts.values()) < 1:\n        raise ValueError(f"Release requires nonempty train/val/test: {dict(counts)}")\n    if len({row["pair_id"] for row in rows}) != len(rows):\n        raise ValueError("Release manifest contains duplicate pair IDs")\n    blocks: dict[str, set[str]] = {}\n    classes: dict[str, Counter] = {split: Counter() for split in counts}\n    regions: dict[str, Counter] = {split: Counter() for split in counts}\n    source_dtypes: set[str] = set()\n    for row in rows:\n        blocks.setdefault(row["spatial_block_id"], set()).add(row["split"])\n        classes[row["split"]][row.get("landscape_class", "unknown")] += 1\n        regions[row["split"]][row.get("geographic_stratum", "unknown")] += 1\n        for field, shape, resolution in (("lr_path", (LR_SIZE, LR_SIZE), 30.0),\n                                         ("hr_path", (HR_SIZE, HR_SIZE), 10.0)):\n            path = resolve(root, row[field])\n            if not path.is_file():\n                raise FileNotFoundError(path)\n            with rasterio.open(path) as source:\n                if (source.height, source.width) != shape or source.count != 4:\n                    raise ValueError(f"Unexpected TIFF contract: {path}")\n                if not np.allclose(source.res, (resolution, resolution), atol=1e-6):\n                    raise ValueError(f"Unexpected resolution: {path}: {source.res}")\n                source_dtypes.update(source.dtypes)\n                _band_indices(source, "rgbnir")\n    leaked = {block: sorted(splits) for block, splits in blocks.items() if len(splits) > 1}\n    if leaked:\n        raise ValueError(f"Spatial blocks cross release splits: {list(leaked.items())[:5]}")\n    protocol_path = root / "protocol.json"\n    protocol = json.loads(protocol_path.read_text(encoding="utf-8")) if protocol_path.is_file() else {}\n    return {\n        "pairs": len(rows), "counts": dict(counts),\n        "realized_fractions": {key: value / len(rows) for key, value in counts.items()},\n        "intended_split_fractions": protocol.get(\n            "release_split_objective",\n            protocol.get("split_fractions", {"train": 0.78, "val": 0.11, "test": 0.11}),\n        ),\n        "split_note": (\n            "The release allocator objective is 78/11/11; complete 100-km "\n            "blocks produce the locked realized counts."\n        ),\n        "geometry": {"lr": [4, LR_SIZE, LR_SIZE], "hr": [4, HR_SIZE, HR_SIZE], "scale": SCALE},\n        "source_band_order": ["blue", "green", "red", "nir"],\n        "model_band_order": list(BAND_ORDER),\n        "source_dtypes": sorted(source_dtypes),\n        "reflectance_decode": ("clip(source/10000,0,1)" if source_dtypes == {"int16"}\n                               else "floating reflectance clipped to [0,1]"),\n        "classes_by_split": {key: dict(value) for key, value in classes.items()},\n        "geographic_strata_by_split": {key: dict(value) for key, value in regions.items()},\n        "release_manifest": str(find_release_manifest(root)),\n        "release_manifest_sha256": digest(find_release_manifest(root)),\n        "protocol_sha256": digest(protocol_path) if protocol_path.is_file() else None,\n        "spatial_blocks": len(blocks), "cross_split_spatial_blocks": 0,\n    }\n\n\ndef save_prediction(path: str | Path, image: np.ndarray, mask: np.ndarray,\n                    reference: str | Path, modality: str, tags: dict) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    names = BAND_ORDER[:3] if modality == "rgb" else BAND_ORDER\n    with rasterio.open(reference) as source:\n        profile = source.profile.copy()\n        metadata = source.tags()\n    profile.update(count=len(names), dtype="float32", compress="deflate", predictor=3, nodata=None)\n    temporary = path.with_suffix(".tmp.tif")\n    with rasterio.Env(GDAL_TIFF_INTERNAL_MASK=True):\n        with rasterio.open(temporary, "w", **profile) as output:\n            output.write(image.astype(np.float32))\n            output.write_mask(mask.squeeze().astype(np.uint8) * 255)\n            metadata.update(sensor="model_prediction", band_order=",".join(names), **tags)\n            output.update_tags(**{key: str(value) for key, value in metadata.items()})\n            if modality == "rgb":\n                output.colorinterp = (ColorInterp.red, ColorInterp.green, ColorInterp.blue)\n            for index, name in enumerate(names, 1):\n                output.set_band_description(index, name)\n    temporary.replace(path)\n', 'metrics.py': '"""Common full-reference metrics for reflectance-domain cross-sensor SR."""\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nfrom torch.nn import functional as F\n\n\ndef _weighted_mean(value: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    if mask.shape[1] == 1 and value.shape[1] != 1:\n        mask = mask.expand(-1, value.shape[1], -1, -1)\n    return (value * mask).sum() / mask.sum().clamp_min(1)\n\n\ndef mae(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    return _weighted_mean((prediction - target).abs(), mask)\n\n\ndef mse(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    return _weighted_mean((prediction - target).square(), mask)\n\n\ndef psnr(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    return -10.0 * torch.log10(mse(prediction, target, mask).clamp_min(1e-12))\n\n\ndef _gaussian_kernel(channels: int, device, dtype) -> torch.Tensor:\n    axis = torch.arange(11, device=device, dtype=dtype) - 5\n    gaussian = torch.exp(-(axis.square()) / (2 * 1.5**2))\n    gaussian = gaussian / gaussian.sum()\n    kernel = torch.outer(gaussian, gaussian)[None, None]\n    return kernel.expand(channels, 1, 11, 11).contiguous()\n\n\ndef ssim(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    channels = prediction.shape[1]\n    kernel = _gaussian_kernel(channels, prediction.device, prediction.dtype)\n    mu_x = F.conv2d(prediction, kernel, padding=5, groups=channels)\n    mu_y = F.conv2d(target, kernel, padding=5, groups=channels)\n    sigma_x = F.conv2d(prediction.square(), kernel, padding=5, groups=channels) - mu_x.square()\n    sigma_y = F.conv2d(target.square(), kernel, padding=5, groups=channels) - mu_y.square()\n    sigma_xy = F.conv2d(prediction * target, kernel, padding=5, groups=channels) - mu_x * mu_y\n    c1, c2 = 0.01**2, 0.03**2\n    value = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / (\n        (mu_x.square() + mu_y.square() + c1) * (sigma_x + sigma_y + c2)\n    ).clamp_min(1e-12)\n    return _weighted_mean(value, mask)\n\n\ndef _correlation(first: torch.Tensor, second: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    if mask.shape[1] == 1 and first.shape[1] != 1:\n        mask = mask.expand(-1, first.shape[1], -1, -1)\n    weight = mask.sum().clamp_min(1)\n    first_mean = (first * mask).sum() / weight\n    second_mean = (second * mask).sum() / weight\n    first_centered = (first - first_mean) * mask\n    second_centered = (second - second_mean) * mask\n    return (first_centered * second_centered).sum() / torch.sqrt(\n        first_centered.square().sum() * second_centered.square().sum()\n    ).clamp_min(1e-12)\n\n\ndef _sobel(image: torch.Tensor) -> torch.Tensor:\n    gray = image[:, :3].mean(dim=1, keepdim=True)\n    x_kernel = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],\n                            device=image.device, dtype=image.dtype)[None, None] / 8\n    y_kernel = x_kernel.transpose(-1, -2)\n    dx = F.conv2d(gray, x_kernel, padding=1)\n    dy = F.conv2d(gray, y_kernel, padding=1)\n    return torch.sqrt(dx.square() + dy.square() + 1e-12)\n\n\ndef edge_f1(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor,\n            threshold: float = 0.04) -> torch.Tensor:\n    predicted = (_sobel(prediction) >= threshold).float() * mask\n    reference = (_sobel(target) >= threshold).float() * mask\n    predicted_dilated = F.max_pool2d(predicted, 3, stride=1, padding=1)\n    reference_dilated = F.max_pool2d(reference, 3, stride=1, padding=1)\n    precision = (predicted * reference_dilated).sum() / predicted.sum().clamp_min(1)\n    recall = (reference * predicted_dilated).sum() / reference.sum().clamp_min(1)\n    return 2 * precision * recall / (precision + recall).clamp_min(1e-12)\n\n\ndef _laplacian(image: torch.Tensor) -> torch.Tensor:\n    channels = image.shape[1]\n    kernel = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]],\n                          device=image.device, dtype=image.dtype)[None, None]\n    return F.conv2d(image, kernel.expand(channels, 1, 3, 3), padding=1, groups=channels)\n\n\ndef _crop(values: tuple[torch.Tensor, ...], border: int) -> tuple[torch.Tensor, ...]:\n    if border == 0:\n        return values\n    return tuple(value[..., border:-border, border:-border] for value in values)\n\n\ndef _y(image: torch.Tensor) -> torch.Tensor:\n    weights = image.new_tensor((0.299, 0.587, 0.114)).view(1, 3, 1, 1)\n    return (image[:, :3] * weights).sum(dim=1, keepdim=True)\n\n\ndef _ndvi(image: torch.Tensor) -> torch.Tensor:\n    red, nir = image[:, 0:1], image[:, 3:4]\n    return (nir - red) / (nir + red).clamp_min(1e-6)\n\n\ndef metric_bundle(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor,\n                  border: int = 3, lpips_model=None) -> dict[str, float]:\n    prediction, target, mask = prediction.float(), target.float(), mask.float()\n    prediction, target, mask = _crop((prediction, target, mask), border)\n    channels = prediction.shape[1]\n    values = {\n        "mae": float(mae(prediction, target, mask)),\n        "rmse": float(torch.sqrt(mse(prediction, target, mask))),\n        "psnr": float(psnr(prediction, target, mask)),\n        "ssim": float(ssim(prediction, target, mask)),\n        "edge_f1": float(edge_f1(prediction, target, mask)),\n        "edge_psnr": float(psnr(_sobel(prediction), _sobel(target), mask)),\n        "gradient_correlation": float(_correlation(_sobel(prediction), _sobel(target), mask)),\n        "scc": float(_correlation(_laplacian(prediction), _laplacian(target), mask)),\n    }\n    rgb_prediction, rgb_target = prediction[:, :3], target[:, :3]\n    values.update(\n        rgb_psnr=float(psnr(rgb_prediction, rgb_target, mask)),\n        rgb_ssim=float(ssim(rgb_prediction, rgb_target, mask)),\n        y_psnr=float(psnr(_y(rgb_prediction), _y(rgb_target), mask)),\n        y_ssim=float(ssim(_y(rgb_prediction), _y(rgb_target), mask)),\n    )\n    names = ("red", "green", "blue", "nir")[:channels]\n    band_mse, band_uiqi, band_means = [], [], []\n    for index, name in enumerate(names):\n        predicted_band, target_band = prediction[:, index:index + 1], target[:, index:index + 1]\n        values[f"{name}_psnr"] = float(psnr(predicted_band, target_band, mask))\n        error = mse(predicted_band, target_band, mask)\n        band_mse.append(error)\n        band_means.append(_weighted_mean(target_band, mask).abs().clamp_min(1e-6))\n        predicted_mean = _weighted_mean(predicted_band, mask)\n        target_mean = _weighted_mean(target_band, mask)\n        predicted_variance = _weighted_mean((predicted_band - predicted_mean).square(), mask)\n        target_variance = _weighted_mean((target_band - target_mean).square(), mask)\n        covariance = _weighted_mean((predicted_band - predicted_mean) * (target_band - target_mean), mask)\n        band_uiqi.append((4 * covariance * predicted_mean * target_mean) /\n                         ((predicted_variance + target_variance) *\n                          (predicted_mean.square() + target_mean.square())).clamp_min(1e-12))\n    values["ergas"] = float((100 / 3) * torch.sqrt(torch.stack([\n        error / mean.square() for error, mean in zip(band_mse, band_means)\n    ]).mean()))\n    values["uiqi"] = float(torch.stack(band_uiqi).mean())\n    dot = (prediction * target).sum(dim=1)\n    denominator = torch.linalg.vector_norm(prediction, dim=1) * torch.linalg.vector_norm(target, dim=1)\n    angle = torch.acos((dot / denominator.clamp_min(1e-8)).clamp(-1 + 1e-7, 1 - 1e-7))\n    values["sam_degrees"] = float(_weighted_mean(angle[:, None], mask) * 180 / math.pi)\n    if channels == 4:\n        predicted_ndvi, target_ndvi = _ndvi(prediction), _ndvi(target)\n        values.update(ndvi_mae=float(mae(predicted_ndvi, target_ndvi, mask)),\n                      ndvi_rmse=float(torch.sqrt(mse(predicted_ndvi, target_ndvi, mask))),\n                      ndvi_correlation=float(_correlation(predicted_ndvi, target_ndvi, mask)))\n    if lpips_model is not None:\n        # Invalid pixels are replaced by the target before the RGB feature comparison.\n        lpips_mask = mask.expand(-1, 3, -1, -1)\n        safe_prediction = rgb_prediction * lpips_mask + rgb_target * (1 - lpips_mask)\n        values["lpips"] = float(lpips_model(safe_prediction * 2 - 1, rgb_target * 2 - 1).mean())\n    return values\n', 'run.py': '"""Train and evaluate pinned x3 SR architectures on the Mahimandal 160 benchmark."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport traceback\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader\n\nfrom adapters import REGISTRY, fetch, load, load_discriminator, predict\nfrom data import LR_SIZE, Pairs, audit, digest, find_release_manifest, resolve, save_prediction\nfrom metrics import mae, metric_bundle, psnr\n\n\ndef write_json(path, value) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef write_csv(path, rows) -> None:\n    if not rows:\n        return\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fields = list(dict.fromkeys(key for row in rows for key in row))\n    with path.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef save_checkpoint(path, state) -> None:\n    temporary = Path(path).with_suffix(".tmp.pt")\n    torch.save(state, temporary)\n    temporary.replace(path)\n\n\ndef canonical(value):\n    return json.loads(json.dumps(value, allow_nan=False))\n\n\ndef synchronize(device: torch.device) -> None:\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n\n\ndef environment(device: torch.device) -> dict:\n    packages = {}\n    for name in ("numpy", "rasterio", "pandas", "matplotlib", "scipy", "lpips", "fvcore"):\n        try:\n            packages[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            packages[name] = None\n    return {"python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda,\n            "device": str(device),\n            "hardware": torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU",\n            "precision": "float32", "packages": packages}\n\n\ndef code_inventory() -> dict:\n    root = Path(__file__).resolve().parent\n    return {name: digest(root / name) for name in\n            ("run.py", "adapters.py", "data.py", "metrics.py", "registry.json")}\n\n\ndef run_root(args) -> Path:\n    return args.output / args.radiometry / args.modality / args.model / f"seed_{args.seed}"\n\n\ndef signature(args, spec: dict) -> dict:\n    release = find_release_manifest(args.dataset)\n    value = {\n        "model": spec, "code_sha256": code_inventory(),\n        "release_manifest_sha256": digest(release), "protocol_sha256": digest(args.dataset / "protocol.json"),\n        "task": "actual Landsat 30 m 160x160 to Sentinel-2 10 m 480x480 paired SR",\n        "scale": 3, "modality": args.modality, "radiometry": args.radiometry,\n        "radiometry_definition": {\n            "reflectance16": "clip(int16/10000,0,1)",\n            "normalized_float32": "clip(float32,0,1)",\n            "quantized8": "round(clip(int16/10000,0,1)*255)/255",\n        }[args.radiometry],\n        "training": {"seed": args.seed, "batch_size": args.batch_size,\n                     "learning_rate": args.lr, "train_lr_crop": args.train_lr_crop,\n                     "loss": ("masked L1" if args.model != "RealESRGAN_SRVGG_X3"\n                              else "masked L1 + RGB LPIPS + RGB logistic GAN after warmup"),\n                     "optimizer": "Adam", "augmentation": "aligned D4",\n                     "sampling": "one random aligned crop per release training pair per epoch",\n                     "model_selection": "highest full-frame validation PSNR"},\n    }\n    return canonical(value)\n\n\ndef strict_load(model: nn.Module, checkpoint: Path):\n    state = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    weights = state.get("model", state.get("state_dict", state)) if isinstance(state, dict) else state\n    if isinstance(weights, nn.Module):\n        weights = weights.state_dict()\n    nn.Module.load_state_dict(model, weights, strict=True)\n    return state\n\n\ndef lpips_network(device: torch.device):\n    import lpips\n    network = lpips.LPIPS(net="alex").to(device).eval()\n    for parameter in network.parameters():\n        parameter.requires_grad_(False)\n    return network\n\n\n@torch.no_grad()\ndef validation(model, spec, dataset, device) -> tuple[float, float]:\n    model.eval()\n    scores = []\n    for lr, hr, mask, _ in DataLoader(dataset, batch_size=1, num_workers=0):\n        sr = predict(model, spec, lr.to(device)).float().clamp(0, 1).cpu()\n        scores.append((float(psnr(sr, hr, mask)), float(mae(sr, hr, mask))))\n    return tuple(np.mean(scores, axis=0).tolist())\n\n\ndef trailing_nonimprovements(history: list[dict]) -> int:\n    best, stale = -float("inf"), 0\n    for row in history:\n        value = row["val_psnr"]\n        if value > best:\n            best, stale = value, 0\n        else:\n            stale += 1\n    return stale\n\n\ndef train(args) -> None:\n    if args.model == "bicubic":\n        raise ValueError("Bicubic has no training stage")\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)\n    device = torch.device(args.device)\n    model, spec = load(args.model, args.sources, args.modality)\n    root = run_root(args)\n    root.mkdir(parents=True, exist_ok=True)\n    receipt = signature(args, spec)\n    protocol_path = root / "protocol.json"\n    if protocol_path.exists() and json.loads(protocol_path.read_text()) != receipt:\n        raise ValueError("Existing run uses different model/data/training settings; choose a new output root")\n    audit_path = args.output / "dataset_audit.json"\n    if not audit_path.exists():\n        write_json(audit_path, audit(args.dataset))\n    previous_audit = json.loads(audit_path.read_text())\n    if previous_audit["release_manifest_sha256"] != receipt["release_manifest_sha256"]:\n        raise ValueError("Dataset release manifest differs from the locked audit")\n    write_json(protocol_path, receipt)\n    (root / "architecture.txt").write_text(str(model), encoding="utf-8")\n    write_json(root / "environment.json", environment(device))\n\n    model = model.to(device)\n    optimizer = torch.optim.Adam((parameter for parameter in model.parameters() if parameter.requires_grad),\n                                 lr=args.lr, betas=(0.9, 0.999))\n    discriminator = discriminator_optimizer = perceptual = None\n    if args.model == "RealESRGAN_SRVGG_X3":\n        discriminator = load_discriminator(args.sources).to(device)\n        discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=args.lr,\n                                                   betas=(0.9, 0.999))\n        perceptual = lpips_network(device)\n    start_epoch, best_psnr, history = 0, -float("inf"), []\n    last_path = root / "last.pt"\n    if last_path.exists():\n        state = strict_load(model, last_path)\n        if state["protocol"] != receipt:\n            raise ValueError("Checkpoint lineage differs from the requested run")\n        optimizer.load_state_dict(state["optimizer"])\n        start_epoch, best_psnr, history = state["epoch"], state["best_psnr"], state["history"]\n        if discriminator is not None:\n            discriminator.load_state_dict(state["discriminator"], strict=True)\n            discriminator_optimizer.load_state_dict(state["discriminator_optimizer"])\n        torch.set_rng_state(state["torch_rng"])\n        np.random.set_state(state["numpy_rng"])\n        random.setstate(state["python_rng"])\n        if device.type == "cuda" and state.get("cuda_rng") is not None:\n            torch.cuda.set_rng_state_all(state["cuda_rng"])\n    stale = trailing_nonimprovements(history)\n    if stale >= args.patience:\n        print(f"Already early-stopped after {stale} non-improving validation epochs", flush=True)\n        return\n    training = Pairs(args.dataset, "train", args.modality, args.radiometry,\n                     augment=True, lr_crop=args.train_lr_crop)\n    validation_set = Pairs(args.dataset, "val", args.modality, args.radiometry)\n    loader_generator = torch.Generator().manual_seed(args.seed)\n    loader = DataLoader(training, batch_size=args.batch_size, shuffle=True,\n                        num_workers=args.workers, generator=loader_generator,\n                        pin_memory=device.type == "cuda")\n    gan_start = max(1, args.epochs // 3)\n    for epoch in range(start_epoch + 1, args.epochs + 1):\n        model.train()\n        losses, pixel_losses, perceptual_losses, adversarial_losses = [], [], [], []\n        epoch_start = time.perf_counter()\n        for lr, hr, mask, _ in loader:\n            lr, hr, mask = lr.to(device), hr.to(device), mask.to(device)\n            sr = predict(model, spec, lr)\n            pixel_loss = mae(sr, hr, mask)\n            perceptual_loss = adversarial_loss = sr.new_zeros(())\n            use_gan = discriminator is not None and epoch > gan_start\n            if use_gan:\n                discriminator.train()\n                discriminator_optimizer.zero_grad(set_to_none=True)\n                real_logits = discriminator(hr[:, :3])\n                fake_logits = discriminator(sr[:, :3].detach())\n                discriminator_loss = F.softplus(-real_logits).mean() + F.softplus(fake_logits).mean()\n                discriminator_loss.backward()\n                discriminator_optimizer.step()\n                for parameter in discriminator.parameters():\n                    parameter.requires_grad_(False)\n                perceptual_loss = perceptual(sr[:, :3] * 2 - 1, hr[:, :3] * 2 - 1).mean()\n                adversarial_loss = F.softplus(-discriminator(sr[:, :3])).mean()\n                for parameter in discriminator.parameters():\n                    parameter.requires_grad_(True)\n            loss = pixel_loss + 0.1 * perceptual_loss + 0.005 * adversarial_loss\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer.step()\n            losses.append(float(loss.detach()))\n            pixel_losses.append(float(pixel_loss.detach()))\n            perceptual_losses.append(float(perceptual_loss.detach()))\n            adversarial_losses.append(float(adversarial_loss.detach()))\n        val_psnr, val_mae = validation(model, spec, validation_set, device)\n        improved = val_psnr > best_psnr\n        best_psnr = max(best_psnr, val_psnr)\n        row = {"epoch": epoch, "loss": float(np.mean(losses)),\n               "pixel_loss": float(np.mean(pixel_losses)),\n               "perceptual_loss": float(np.mean(perceptual_losses)),\n               "adversarial_loss": float(np.mean(adversarial_losses)),\n               "val_psnr": val_psnr, "val_mae": val_mae,\n               "seconds": time.perf_counter() - epoch_start, "improved": improved}\n        history.append(row)\n        state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),\n                 "discriminator": discriminator.state_dict() if discriminator is not None else None,\n                 "discriminator_optimizer": (discriminator_optimizer.state_dict()\n                                             if discriminator_optimizer is not None else None),\n                 "epoch": epoch, "best_psnr": best_psnr, "history": history,\n                 "protocol": receipt, "torch_rng": torch.get_rng_state(),\n                 "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,\n                 "numpy_rng": np.random.get_state(), "python_rng": random.getstate()}\n        save_checkpoint(last_path, state)\n        if improved:\n            save_checkpoint(root / "best.pt", state)\n        write_csv(root / "training.csv", history)\n        stale = trailing_nonimprovements(history)\n        print(f"[{args.model}/{args.modality}/{args.radiometry}] epoch {epoch}/{args.epochs} "\n              f"loss={row[\'loss\']:.6f} val_psnr={val_psnr:.4f} stale={stale}/{args.patience}", flush=True)\n        if stale >= args.patience:\n            break\n    write_json(root / "training_status.json",\n               {"last_completed_epoch": history[-1]["epoch"], "requested_epoch_ceiling": args.epochs,\n                "early_stopped": trailing_nonimprovements(history) >= args.patience,\n                "patience": args.patience, "best_validation_psnr": best_psnr,\n                "best_checkpoint_sha256": digest(root / "best.pt")})\n\n\ndef _flops(model, sample: torch.Tensor) -> tuple[float | None, str]:\n    if model is None:\n        return None, "not_applicable"\n    try:\n        from fvcore.nn import FlopCountAnalysis\n        return float(FlopCountAnalysis(model, (sample,)).total()), "fvcore_operator_count"\n    except Exception as error:\n        return None, f"unavailable: {type(error).__name__}: {error}"\n\n\n@torch.no_grad()\ndef evaluate(args) -> None:\n    device = torch.device(args.device)\n    model, spec = load(args.model, args.sources, args.modality)\n    root = run_root(args)\n    checkpoint = args.checkpoint or root / "best.pt"\n    checkpoint_hash = None\n    if model is not None:\n        if not checkpoint.is_file():\n            raise FileNotFoundError(f"Train first or supply --checkpoint: {checkpoint}")\n        state = strict_load(model, checkpoint)\n        checkpoint_hash = digest(checkpoint)\n        expected = signature(args, spec)\n        if state.get("protocol") != expected:\n            raise ValueError("Checkpoint does not match the requested dataset/modality/radiometry protocol")\n        model = model.to(device).eval()\n    dataset = Pairs(args.dataset, args.split, args.modality, args.radiometry)\n    audit_info = audit(args.dataset)\n    evaluation_id = checkpoint_hash[:12] if checkpoint_hash else "bicubic"\n    destination = root / "evaluation" / args.split / evaluation_id\n    receipt = canonical({"model": spec, "checkpoint_sha256": checkpoint_hash,\n                         "code_sha256": code_inventory(), "dataset": audit_info,\n                         "split": args.split, "modality": args.modality,\n                         "radiometry": args.radiometry, "border_crop": args.border_crop,\n                         "test_time_augmentation": False, "inference": "whole 160x160 frame, float32"})\n    destination.mkdir(parents=True, exist_ok=True)\n    protocol_path = destination / "protocol.json"\n    if protocol_path.exists() and json.loads(protocol_path.read_text()) != receipt:\n        raise ValueError("Existing evaluation protocol differs")\n    write_json(protocol_path, receipt)\n    write_json(destination / "environment.json", environment(device))\n    if (destination / "metrics.json").exists() and (destination / "per_image.csv").exists():\n        print("Completed evaluation reused:", destination, flush=True)\n        return\n    perceptual = lpips_network(device)\n    sample = dataset[0][0][None].to(device)\n    for _ in range(3):\n        predict(model, spec, sample)\n    synchronize(device)\n    if device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats(device)\n    flop_count, flop_status = _flops(model, sample)\n    rows, inference_times = [], []\n    started = time.perf_counter()\n    for index, (lr, hr, mask, _) in enumerate(DataLoader(dataset, batch_size=1, num_workers=0)):\n        record = dataset.rows[index]\n        lr_device = lr.to(device)\n        synchronize(device)\n        tick = time.perf_counter()\n        sr_device = predict(model, spec, lr_device)\n        synchronize(device)\n        milliseconds = (time.perf_counter() - tick) * 1000\n        sr_device = sr_device.float().clamp(0, 1)\n        if not torch.isfinite(sr_device).all():\n            raise FloatingPointError(record["pair_id"])\n        values = metric_bundle(sr_device, hr.to(device), mask.to(device),\n                               border=args.border_crop, lpips_model=perceptual)\n        rows.append({"pair_id": record["pair_id"], "model": args.model,\n                     "modality": args.modality, "radiometry": args.radiometry,\n                     "landscape_class": record.get("landscape_class", ""),\n                     "geographic_stratum": record.get("geographic_stratum", ""),\n                     "state_or_ut": record.get("state_or_ut", ""),\n                     "priority_region": record.get("priority_region", ""),\n                     "spatial_block_id": record.get("spatial_block_id", ""),\n                     "day_gap": record.get("day_gap", ""),\n                     "inference_ms": milliseconds, **values})\n        inference_times.append(milliseconds)\n        sr = sr_device.cpu()\n        save_prediction(destination / "predictions" / f"{record[\'pair_id\']}.tif",\n                        sr[0].numpy(), mask[0].numpy() > 0,\n                        resolve(args.dataset, record["hr_path"]), args.modality,\n                        {"model": args.model, "checkpoint_sha256": checkpoint_hash or "none",\n                         "radiometry": args.radiometry, "border_crop_for_metrics": args.border_crop})\n        if (index + 1) % 25 == 0 or index + 1 == len(dataset):\n            print(f"[{args.model}/{args.split}] {index + 1}/{len(dataset)} "\n                  f"PSNR={np.mean([row[\'psnr\'] for row in rows]):.4f}", flush=True)\n    metric_names = [key for key in rows[0] if key not in {\n        "pair_id", "model", "modality", "radiometry", "landscape_class",\n        "geographic_stratum", "state_or_ut", "priority_region", "spatial_block_id", "day_gap"\n    }]\n    summary = {key: float(np.mean([row[key] for row in rows])) for key in metric_names}\n    timing = np.asarray(inference_times)\n    summary.update({"model": args.model, "modality": args.modality,\n                    "radiometry": args.radiometry, "split": args.split, "count": len(rows),\n                    "seed": args.seed, "parameters": spec.get("parameters", 0),\n                    "flops_160": flop_count, "flops_status": flop_status,\n                    "seconds_total": time.perf_counter() - started,\n                    "inference_mean_ms": float(timing.mean()),\n                    "inference_median_ms": float(np.median(timing)),\n                    "inference_p95_ms": float(np.percentile(timing, 95)),\n                    "images_per_second": 1000 / float(timing.mean()),\n                    "peak_allocated_mb": (torch.cuda.max_memory_allocated(device) / 2**20\n                                          if device.type == "cuda" else None),\n                    "metric_domain": "clipped reflectance [0,1]",\n                    "border_crop": args.border_crop,\n                    "lpips_domain": "RGB AlexNet features; invalid pixels replaced with target"})\n    write_csv(destination / "per_image.csv", rows)\n    write_json(destination / "per_image.json", rows)\n    write_json(destination / "metrics.json", summary)\n    print(json.dumps(summary, indent=2), flush=True)\n\n\ndef smoke(args) -> None:\n    model, spec = load(args.model, args.sources, args.modality)\n    device = torch.device(args.device)\n    if model is not None:\n        model = model.to(device).train(args.backward)\n    channels = 3 if args.modality == "rgb" else 4\n    value = torch.rand(1, channels, LR_SIZE, LR_SIZE, device=device)\n    with torch.set_grad_enabled(args.backward):\n        output = predict(model, spec, value)\n        if args.backward and model is not None:\n            output.square().mean().backward()\n    if tuple(output.shape) != (1, channels, 480, 480) or not torch.isfinite(output).all():\n        raise RuntimeError(f"Invalid preflight output: {tuple(output.shape)}")\n    report = {"status": "passed", "shape": list(output.shape), "backward": args.backward, **spec}\n    write_json(args.output / "preflight" / f"{args.model}_{args.modality}.json", report)\n    print(json.dumps(report, indent=2), flush=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=("fetch", "audit", "smoke", "train", "evaluate"))\n    parser.add_argument("--sources", type=Path, default=None)\n    parser.add_argument("--model", choices=[*REGISTRY, "bicubic"], default="EDSR")\n    parser.add_argument("--dataset", type=Path)\n    parser.add_argument("--output", type=Path, default=None)\n    parser.add_argument("--modality", choices=("rgb", "rgbnir"), default="rgb")\n    parser.add_argument("--radiometry", choices=("reflectance16", "normalized_float32", "quantized8"),\n                        default="reflectance16")\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--patience", type=int, default=3)\n    parser.add_argument("--batch-size", type=int, default=1)\n    parser.add_argument("--train-lr-crop", type=int, default=64)\n    parser.add_argument("--lr", type=float, default=1e-4)\n    parser.add_argument("--workers", type=int, default=0)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--split", choices=("val", "test"), default="val")\n    parser.add_argument("--checkpoint", type=Path)\n    parser.add_argument("--border-crop", type=int, default=3)\n    parser.add_argument("--backward", action="store_true")\n    args = parser.parse_args()\n    torch.set_num_threads(min(4, os.cpu_count() or 1))\n    if min(args.epochs, args.patience, args.batch_size, args.train_lr_crop) < 1:\n        parser.error("epochs, patience, batch size and crop must be positive")\n    if args.border_crop < 0 or args.border_crop >= 240:\n        parser.error("invalid border crop")\n    if args.action == "fetch":\n        fetch(args.sources)\n        return\n    if args.action == "smoke":\n        smoke(args)\n        return\n    if args.dataset is None:\n        parser.error("--dataset is required")\n    if args.action == "audit":\n        result = audit(args.dataset)\n        write_json(args.output / "dataset_audit.json", result)\n        print(json.dumps(result, indent=2))\n        return\n    try:\n        (train if args.action == "train" else evaluate)(args)\n    except Exception:\n        failure = run_root(args) / f"{args.action}_failure.json"\n        write_json(failure, {"status": "failed", "model": args.model,\n                             "modality": args.modality, "radiometry": args.radiometry,\n                             "error": traceback.format_exc()})\n        raise\n\n\nif __name__ == "__main__":\n    main()\n', 'report.py': '"""Paper tables, paired inference, qualitative panels and evidence bundles."""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nimport zipfile\nfrom datetime import datetime, timezone\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport rasterio\n\nfrom data import Pairs, digest\n\n\nIDENTITY = ["radiometry", "modality", "model", "seed"]\nHIGHER = {"psnr", "ssim", "rgb_psnr", "rgb_ssim", "y_psnr", "y_ssim",\n          "edge_f1", "gradient_correlation", "scc", "uiqi", "ndvi_correlation"}\nLOWER = {"mae", "rmse", "ergas", "sam_degrees", "lpips", "ndvi_mae", "ndvi_rmse"}\n\n\ndef _identity(path: Path, root: Path) -> dict:\n    relative = path.relative_to(root).parts\n    seed_component = relative[3]\n    if not seed_component.startswith("seed_"):\n        raise ValueError(f"Invalid run seed directory: {seed_component}")\n    return {"radiometry": relative[0], "modality": relative[1], "model": relative[2],\n            "seed": int(seed_component[len("seed_"):]), "split": relative[5],\n            "evaluation_id": relative[6]}\n\n\ndef _bootstrap(values: np.ndarray, clusters: np.ndarray | None = None,\n               repeats: int = 2000, seed: int = 42) -> tuple[float, float]:\n    rng = np.random.default_rng(seed)\n    estimates = []\n    if clusters is None:\n        for _ in range(repeats):\n            estimates.append(values[rng.integers(0, len(values), len(values))].mean())\n    else:\n        groups = np.unique(clusters)\n        index = {group: np.flatnonzero(clusters == group) for group in groups}\n        for _ in range(repeats):\n            selected = rng.choice(groups, size=len(groups), replace=True)\n            estimates.append(np.concatenate([values[index[group]] for group in selected]).mean())\n    return tuple(np.percentile(estimates, (2.5, 97.5)).tolist())\n\n\ndef collect(root: str | Path, split: str = "test") -> tuple[pd.DataFrame, pd.DataFrame]:\n    root = Path(root)\n    summaries, records = [], []\n    for metrics_path in root.glob(f"*/*/*/seed_*/evaluation/{split}/*/metrics.json"):\n        identity = _identity(metrics_path, root)\n        summaries.append({**identity, **json.loads(metrics_path.read_text())})\n        per_image = metrics_path.with_name("per_image.csv")\n        if per_image.is_file():\n            frame = pd.read_csv(per_image)\n            for key, value in identity.items():\n                frame[key] = value\n            frame["evaluation_root"] = str(metrics_path.parent)\n            records.append(frame)\n    if not summaries:\n        raise FileNotFoundError(f"No {split} metrics below {root}")\n    return pd.DataFrame(summaries), pd.concat(records, ignore_index=True)\n\n\ndef paired_tables(per_image: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    metrics = sorted((HIGHER | LOWER).intersection(per_image.columns))\n    for key, frame in per_image.groupby(["radiometry", "modality", "seed"], dropna=False):\n        baseline = frame[frame.model == "bicubic"].set_index("pair_id")\n        if baseline.empty:\n            continue\n        for model, candidate in frame[frame.model != "bicubic"].groupby("model"):\n            candidate = candidate.set_index("pair_id")\n            common = baseline.index.intersection(candidate.index)\n            if not len(common):\n                continue\n            clusters = candidate.loc[common, "spatial_block_id"].astype(str).to_numpy()\n            for metric in metrics:\n                direction = 1 if metric in HIGHER else -1\n                improvement = direction * (candidate.loc[common, metric].to_numpy(dtype=float) -\n                                            baseline.loc[common, metric].to_numpy(dtype=float))\n                patch_low, patch_high = _bootstrap(improvement)\n                cluster_low, cluster_high = _bootstrap(improvement, clusters)\n                rows.append({"radiometry": key[0], "modality": key[1], "seed": key[2],\n                             "model": model, "metric": metric,\n                             "direction": "higher" if direction == 1 else "lower",\n                             "mean_improvement_vs_bicubic": improvement.mean(),\n                             "patch_ci95_low": patch_low, "patch_ci95_high": patch_high,\n                             "spatial_block_ci95_low": cluster_low,\n                             "spatial_block_ci95_high": cluster_high,\n                             "fraction_better": float((improvement > 0).mean()),\n                             "count": len(improvement), "spatial_blocks": len(np.unique(clusters))})\n    return pd.DataFrame(rows)\n\n\ndef grouped_tables(per_image: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:\n    metrics = [name for name in ("psnr", "ssim", "mae", "sam_degrees", "lpips",\n                                  "edge_f1", "ndvi_rmse", "ndvi_correlation")\n               if name in per_image]\n    keys = ["radiometry", "modality", "model", "seed"]\n    by_class = per_image.groupby(keys + ["landscape_class"], dropna=False)[metrics].agg(["mean", "std", "count"])\n    by_class.columns = ["_".join(column) for column in by_class.columns]\n    by_region = per_image.groupby(keys + ["geographic_stratum"], dropna=False)[metrics].agg(["mean", "std", "count"])\n    by_region.columns = ["_".join(column) for column in by_region.columns]\n    return by_class.reset_index(), by_region.reset_index()\n\n\ndef ablation_tables(summary: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:\n    rgb_rows, quantization_rows = [], []\n    for (model, seed), frame in summary.groupby(["model", "seed"]):\n        reference = frame[frame.radiometry == "reflectance16"]\n        rgb = reference[reference.modality == "rgb"]\n        rgbnir = reference[reference.modality == "rgbnir"]\n        if len(rgb) == 1 and len(rgbnir) == 1:\n            rgb_rows.append({"model": model, "seed": seed,\n                             "rgb_only_rgb_psnr": float(rgb.iloc[0].rgb_psnr),\n                             "rgbnir_rgb_psnr": float(rgbnir.iloc[0].rgb_psnr),\n                             "rgb_psnr_delta_with_nir": float(rgbnir.iloc[0].rgb_psnr - rgb.iloc[0].rgb_psnr),\n                             "rgb_only_rgb_ssim": float(rgb.iloc[0].rgb_ssim),\n                             "rgbnir_rgb_ssim": float(rgbnir.iloc[0].rgb_ssim),\n                             "rgb_ssim_delta_with_nir": float(rgbnir.iloc[0].rgb_ssim - rgb.iloc[0].rgb_ssim),\n                             "ndvi_rmse": float(rgbnir.iloc[0].get("ndvi_rmse", np.nan))})\n        for modality, subset in frame.groupby("modality"):\n            full = subset[subset.radiometry == "reflectance16"]\n            quantized = subset[subset.radiometry == "quantized8"]\n            if len(full) == 1 and len(quantized) == 1:\n                quantization_rows.append({"model": model, "seed": seed, "modality": modality,\n                                          "reflectance16_psnr": float(full.iloc[0].psnr),\n                                          "quantized8_psnr": float(quantized.iloc[0].psnr),\n                                          "quantized_minus_reflectance_psnr": float(quantized.iloc[0].psnr - full.iloc[0].psnr),\n                                          "reflectance16_sam": float(full.iloc[0].sam_degrees),\n                                          "quantized8_sam": float(quantized.iloc[0].sam_degrees)})\n    return pd.DataFrame(rgb_rows), pd.DataFrame(quantization_rows)\n\n\ndef _plot_summary(summary: pd.DataFrame, destination: Path) -> None:\n    subset = summary[(summary.radiometry == "reflectance16") & (summary.modality == "rgb")]\n    if subset.empty:\n        return\n    subset = subset.sort_values("psnr", ascending=False)\n    figure, axes = plt.subplots(2, 2, figsize=(15, 10))\n    for axis, metric, title in zip(axes.flat, ("psnr", "ssim", "lpips", "inference_mean_ms"),\n                                   ("PSNR (dB)", "SSIM", "LPIPS", "Inference latency (ms)")):\n        if metric not in subset:\n            axis.axis("off")\n            continue\n        axis.barh(subset.model, subset[metric], color="#315b7d")\n        axis.invert_yaxis()\n        axis.set_title(title)\n        axis.grid(axis="x", alpha=0.25)\n    figure.suptitle("Mahimandal 160→480 test benchmark: reflectance16 RGB")\n    figure.tight_layout()\n    figure.savefig(destination, dpi=180)\n    plt.close(figure)\n\n\ndef summarize(root: str | Path, split: str = "test") -> pd.DataFrame:\n    root = Path(root)\n    summary, per_image = collect(root, split)\n    report = root / "reports" / split\n    report.mkdir(parents=True, exist_ok=True)\n    summary.sort_values(["radiometry", "modality", "psnr"], ascending=[True, True, False]).to_csv(\n        report / "all_runs.csv", index=False)\n    per_image.to_csv(report / "all_per_image.csv", index=False)\n    paired_tables(per_image).to_csv(report / "paired_vs_bicubic.csv", index=False)\n    by_class, by_region = grouped_tables(per_image)\n    by_class.to_csv(report / "by_landscape_class.csv", index=False)\n    by_region.to_csv(report / "by_geographic_stratum.csv", index=False)\n    nir, quantization = ablation_tables(summary)\n    nir.to_csv(report / "rgb_vs_rgbnir.csv", index=False)\n    quantization.to_csv(report / "reflectance16_vs_quantized8.csv", index=False)\n    complexity_columns = [column for column in ("model", "modality", "parameters", "flops_160",\n                                                 "inference_mean_ms", "inference_p95_ms",\n                                                 "images_per_second", "peak_allocated_mb")\n                          if column in summary]\n    summary[complexity_columns].drop_duplicates().to_csv(report / "complexity.csv", index=False)\n    for (radiometry, modality), frame in summary.groupby(["radiometry", "modality"]):\n        ordered = frame.sort_values("psnr", ascending=False)\n        ordered.to_csv(\n            report / f"main_{radiometry}_{modality}.csv", index=False)\n        paper_columns = [column for column in\n                         ("model", "count", "psnr", "ssim", "mae", "rmse", "edge_f1",\n                          "ergas", "sam_degrees", "uiqi", "scc", "lpips", "ndvi_rmse",\n                          "ndvi_correlation", "parameters", "flops_160",\n                          "inference_mean_ms", "peak_allocated_mb") if column in ordered]\n        ordered[paper_columns].to_latex(report / f"main_{radiometry}_{modality}.tex",\n                                        index=False, float_format="%.4f")\n    _plot_summary(summary, report / "main_metrics.png")\n    inventory = {"split": split, "runs": len(summary), "per_image_rows": len(per_image),\n                 "models": sorted(summary.model.unique().tolist()),\n                 "modalities": sorted(summary.modality.unique().tolist()),\n                 "radiometry": sorted(summary.radiometry.unique().tolist()),\n                 "warning": "Do not select models or hyperparameters using test results."}\n    (report / "report_inventory.json").write_text(json.dumps(inventory, indent=2), encoding="utf-8")\n    return summary\n\n\ndef _rgb(array: np.ndarray, display_max: float = 0.3) -> np.ndarray:\n    return np.clip(array[:3].transpose(1, 2, 0) / display_max, 0, 1)\n\n\ndef visualize(root: str | Path, dataset_root: str | Path, index: int = 0,\n              radiometry: str = "reflectance16", modality: str = "rgb",\n              seed: int = 42, split: str = "test", models: list[str] | None = None,\n              display_max: float = 0.3) -> Path:\n    root, dataset_root = Path(root), Path(dataset_root)\n    dataset = Pairs(dataset_root, split, modality, radiometry)\n    lr, hr, _, _ = dataset[index]\n    record = dataset.rows[index]\n    predictions = []\n    for metrics_path in root.glob(f"{radiometry}/{modality}/*/seed_{seed}/evaluation/{split}/*/metrics.json"):\n        model = metrics_path.relative_to(root).parts[2]\n        if models is not None and model not in models:\n            continue\n        path = metrics_path.parent / "predictions" / f"{record[\'pair_id\']}.tif"\n        if path.is_file():\n            with rasterio.open(path) as source:\n                predictions.append((model, source.read()))\n    lr_up = np.repeat(np.repeat(lr.numpy(), 3, axis=1), 3, axis=2)\n    panels = [("Landsat LR (nearest display)", lr_up),\n              ("Sentinel-2 target", hr.numpy())] + sorted(predictions)\n    target = hr.numpy()\n    columns = len(panels)\n    figure, axes = plt.subplots(3, columns, figsize=(3.25 * columns, 9.5), squeeze=False)\n    height, width = target.shape[-2:]\n    zoom = (slice(height // 3, 2 * height // 3), slice(width // 3, 2 * width // 3))\n    for column, (title, image) in enumerate(panels):\n        axes[0, column].imshow(_rgb(image, display_max), interpolation="nearest")\n        axes[0, column].set_title(title, fontsize=9)\n        axes[1, column].imshow(_rgb(image[:, zoom[0], zoom[1]], display_max), interpolation="nearest")\n        error = np.mean(np.abs(image[:3] - target[:3]), axis=0)\n        axes[2, column].imshow(error, cmap="magma", vmin=0, vmax=0.10, interpolation="nearest")\n        for row in range(3):\n            axes[row, column].axis("off")\n    axes[0, 0].set_ylabel("Full frame")\n    axes[1, 0].set_ylabel("Center zoom")\n    axes[2, 0].set_ylabel("RGB absolute error")\n    figure.suptitle(f"{split}/{index} | {record[\'pair_id\']} | {record.get(\'landscape_class\', \'\')}\\n"\n                    f"Shared display stretch [0,{display_max}]; never used for metrics")\n    figure.tight_layout()\n    destination = root / "reports" / split / "qualitative"\n    destination.mkdir(parents=True, exist_ok=True)\n    path = destination / f"{radiometry}_{modality}_{index}.png"\n    figure.savefig(path, dpi=180)\n    plt.close(figure)\n    return path\n\n\ndef bundle(root: str | Path, include_predictions: bool = False) -> Path:\n    root = Path(root)\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")\n    suffix = "full_predictions" if include_predictions else "paper_results"\n    destination = root.parent / f"{root.name}_{suffix}_{stamp}.zip"\n    allowed = {".json", ".csv", ".tex", ".txt", ".png", ".pdf"}\n    if include_predictions:\n        allowed.add(".tif")\n    with zipfile.ZipFile(destination, "x", zipfile.ZIP_DEFLATED, compresslevel=3) as archive:\n        inventory = []\n        for path in root.rglob("*"):\n            if not path.is_file() or path.suffix not in allowed:\n                continue\n            if not include_predictions and "predictions" in path.parts:\n                continue\n            relative = path.relative_to(root).as_posix()\n            archive.write(path, relative)\n            inventory.append(f"{digest(path)}  {relative}")\n        archive.writestr("checksums.sha256", "\\n".join(inventory) + "\\n")\n    return destination\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("root", type=Path)\n    parser.add_argument("--split", choices=("val", "test"), default="test")\n    parser.add_argument("--bundle", action="store_true")\n    args = parser.parse_args()\n    print(summarize(args.root, args.split).to_string(index=False))\n    if args.bundle:\n        print(bundle(args.root))\n', 'qualitative_gallery.py': '"""Deterministic, complete-model qualitative gallery from saved test predictions.\n\nThis reads existing evaluation GeoTIFFs. It never runs inference or changes\ncheckpoints, predictions, the dataset, or the locked evaluation protocol.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport rasterio\n\nfrom data import Pairs, digest, find_release_manifest\n\n\ndef evaluation_dirs(root, radiometry, modality, seed, split, border_crop):\n    base = root / radiometry / modality\n    found = {}\n    for metrics in sorted(base.glob(f"*/seed_{seed}/evaluation/{split}/*/metrics.json")):\n        directory = metrics.parent\n        protocol_path = directory / "protocol.json"\n        if not protocol_path.is_file():\n            continue\n        protocol = json.loads(protocol_path.read_text(encoding="utf-8"))\n        if (protocol.get("border_crop") != border_crop or\n                protocol.get("radiometry") != radiometry or\n                protocol.get("modality") != modality or\n                protocol.get("split") != split):\n            continue\n        model = directory.relative_to(root).parts[2]\n        if model in found:\n            raise ValueError(f"Ambiguous {model} evaluation: {found[model]} and {directory}")\n        found[model] = (directory, protocol)\n    if "bicubic" not in found:\n        raise FileNotFoundError(f"No matching bicubic evaluation in {base}")\n    return dict(sorted(found.items(), key=lambda item: (item[0] != "bicubic", item[0].casefold())))\n\n\ndef _complete_prediction_paths(pair_id, evaluations):\n    paths = {}\n    for modality, models in evaluations.items():\n        paths[modality] = {}\n        for name, (directory, _) in models.items():\n            path = directory / "predictions" / f"{pair_id}.tif"\n            if not path.is_file():\n                return None\n            paths[modality][name] = path\n    return paths\n\n\ndef select_cases(rows, evaluations, group_by, per_group, seed):\n    eligible = []\n    for index, row in enumerate(rows):\n        paths = _complete_prediction_paths(row["pair_id"], evaluations)\n        if paths is not None:\n            eligible.append((index, row, paths))\n    if not eligible:\n        raise FileNotFoundError("No test pair has predictions from every evaluated model")\n    groups = defaultdict(list)\n    expected = defaultdict(int)\n    for row in rows:\n        for field in group_by:\n            expected[(field, row.get(field, "unknown") or "unknown")] += 1\n    for item in eligible:\n        _, row, _ = item\n        for field in group_by:\n            groups[(field, row.get(field, "unknown") or "unknown")].append(item)\n    selections = []\n    for (field, label), total in sorted(expected.items()):\n        candidates = groups.get((field, label), [])\n        if len(candidates) < per_group:\n            raise ValueError(\n                f"{field}={label}: need {per_group} pairs with ALL model predictions; "\n                f"found {len(candidates)} of {total}. The compact bundle cannot "\n                "make three distinct examples per group; use full DGX output."\n            )\n        shuffled = sorted(candidates, key=lambda item: hashlib.sha256(\n            f"{seed}:{item[1][\'pair_id\']}".encode("utf-8")\n        ).digest())\n        chosen = []\n        for _ in range(per_group):\n            used_blocks = {item[1].get("spatial_block_id") for item in chosen}\n            used_states = {item[1].get("state_or_ut") for item in chosen}\n            remaining = [item for item in shuffled if item not in chosen]\n            best = max(remaining, key=lambda item: (\n                item[1].get("spatial_block_id") not in used_blocks,\n                item[1].get("state_or_ut") not in used_states,\n            ))\n            chosen.append(best)\n        for rank, (index, row, paths) in enumerate(chosen, 1):\n            selections.append((field, label, rank, index, row, paths))\n    return selections, len(eligible), expected\n\n\ndef _rgb(array, display_max):\n    return np.clip(array[:3].transpose(1, 2, 0) / display_max, 0.0, 1.0)\n\n\ndef render_case(dataset, index, predictions, output_path, title, display_max,\n                error_max, dpi):\n    lr, hr, valid_mask, _ = dataset[index]\n    target = hr.numpy()\n    valid = valid_mask.numpy()[0] > 0\n    lr_up = np.repeat(np.repeat(lr.numpy(), 3, axis=1), 3, axis=2)\n    panels = [("Landsat LR\\n(nearest display)", lr_up),\n              ("Sentinel-2\\ntarget", target)]\n    for name, path in predictions.items():\n        with rasterio.open(path) as source:\n            image = source.read()\n        if image.shape != target.shape or not np.isfinite(image).all():\n            raise ValueError(f"Invalid prediction shape/values: {path}: {image.shape}")\n        panels.append((name.replace("_", "\\n", 1), image))\n    columns = len(panels)\n    figure, axes = plt.subplots(3, columns, figsize=(2.55 * columns, 7.7), squeeze=False)\n    height, width = target.shape[-2:]\n    zoom = (slice(height // 3, 2 * height // 3),\n            slice(width // 3, 2 * width // 3))\n    for column, (name, image) in enumerate(panels):\n        axes[0, column].imshow(_rgb(image, display_max), interpolation="nearest")\n        axes[0, column].set_title(name, fontsize=8)\n        axes[1, column].imshow(_rgb(image[:, zoom[0], zoom[1]], display_max),\n                               interpolation="nearest")\n        if column == 1:\n            error = np.zeros((height, width), dtype=np.float32)\n        else:\n            error = np.mean(np.abs(image[:3] - target[:3]), axis=0)\n        error = np.where(valid, error, np.nan)\n        axes[2, column].imshow(error, cmap="magma", vmin=0, vmax=error_max,\n                               interpolation="nearest")\n        for row_index in range(3):\n            axes[row_index, column].set_xticks([])\n            axes[row_index, column].set_yticks([])\n    for row_index, text in enumerate(("Full", "Center zoom", "RGB |error|")):\n        axes[row_index, 0].set_ylabel(text, fontsize=9)\n    figure.suptitle(title + f"\\nShared RGB display [0,{display_max:g}]; "\n                    f"shared error scale [0,{error_max:g}]. Neither affects metrics.",\n                    fontsize=10)\n    figure.tight_layout(rect=(0, 0, 1, 0.94))\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    figure.savefig(output_path, dpi=dpi, facecolor="white")\n    plt.close(figure)\n\n\ndef _write_contact_sheet(items, destination, label, display_max, dpi):\n    figure, axes = plt.subplots(2, len(items), figsize=(3.7 * len(items), 7.8),\n                               squeeze=False)\n    for column, (rank, image, row) in enumerate(items):\n        lr, hr, _, _ = image\n        axes[0, column].imshow(_rgb(hr.numpy(), display_max))\n        axes[1, column].imshow(_rgb(np.repeat(np.repeat(lr.numpy(), 3, 1), 3, 2),\n                                    display_max), interpolation="nearest")\n        axes[0, column].set_title(f"Candidate {rank}: {row.get(\'state_or_ut\', \'\')}\\n"\n                                  f"{row[\'pair_id\']}", fontsize=8)\n        for r in (0, 1):\n            axes[r, column].axis("off")\n    axes[0, 0].set_ylabel("Sentinel-2 HR")\n    axes[1, 0].set_ylabel("Landsat LR")\n    figure.suptitle(label + " | source pairs only; model panels in individual PNGs")\n    figure.tight_layout(rect=(0, 0, 1, 0.96))\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    figure.savefig(destination, dpi=dpi, facecolor="white")\n    plt.close(figure)\n\n\ndef build_gallery(output, dataset_root, destination, archive, group_by,\n                  per_group=3, seed=42, split="test", radiometry="reflectance16",\n                  modalities=("rgb", "rgbnir"), border_crop=3, display_max=0.3,\n                  error_max=0.10, dpi=110, verify_manifest=True):\n    output, dataset_root = Path(output).resolve(), Path(dataset_root).resolve()\n    destination, archive = Path(destination).resolve(), Path(archive).resolve()\n    if destination == output or output in destination.parents:\n        raise ValueError("Gallery destination must be outside OUTPUT to avoid recursive bundling")\n    if archive == output or output in archive.parents:\n        raise ValueError("Gallery archive must be outside OUTPUT")\n    if not output.is_dir() or not dataset_root.is_dir():\n        raise FileNotFoundError("OUTPUT and DATASET_ROOT must both exist")\n    manifest_sha = digest(find_release_manifest(dataset_root))\n    evaluations = {modality: evaluation_dirs(output, radiometry, modality, seed,\n                                              split, border_crop)\n                   for modality in modalities}\n    mismatches = []\n    for modality, models in evaluations.items():\n        for name, (_, protocol) in models.items():\n            recorded = protocol.get("dataset", {}).get("release_manifest_sha256")\n            if recorded != manifest_sha:\n                mismatches.append((modality, name, recorded))\n    if mismatches and verify_manifest:\n        raise ValueError(f"Dataset manifest SHA mismatch; refusing mixed data: {mismatches[:3]}. "\n                         f"Actual SHA256={manifest_sha}")\n    datasets = {modality: Pairs(dataset_root, split, modality, radiometry)\n                for modality in modalities}\n    rows = datasets[modalities[0]].rows\n    for dataset in datasets.values():\n        if [row["pair_id"] for row in dataset.rows] != [row["pair_id"] for row in rows]:\n            raise ValueError("Modality datasets do not have the same pair order")\n    selections, eligible_count, expected = select_cases(\n        rows, evaluations, group_by, per_group, seed)\n    destination.mkdir(parents=True, exist_ok=True)\n    inventory = []\n    sheets = defaultdict(list)\n    for field, label, rank, index, row, paths in selections:\n        safe_label = "".join(c if c.isalnum() or c in "_-" else "_" for c in label)\n        basename = f"{rank:02d}_{row[\'pair_id\']}"\n        title = (f"{split}/{index} | {row[\'pair_id\']} | {row.get(\'landscape_class\', \'\')} | "\n                 f"{row.get(\'geographic_stratum\', \'\')} | {row.get(\'state_or_ut\', \'\')}")\n        for modality in modalities:\n            path = destination / field / safe_label / modality / f"{basename}.png"\n            render_case(datasets[modality], index, paths[modality], path, title,\n                        display_max, error_max, dpi)\n            inventory.append({\n                "group_by": field, "group": label, "candidate": rank,\n                "modality": modality, "pair_id": row["pair_id"], "test_index": index,\n                "landscape_class": row.get("landscape_class", ""),\n                "geographic_stratum": row.get("geographic_stratum", ""),\n                "state_or_ut": row.get("state_or_ut", ""),\n                "spatial_block_id": row.get("spatial_block_id", ""),\n                "center_lon": row.get("center_lon", ""),\n                "center_lat": row.get("center_lat", ""),\n                "model_count": len(paths[modality]),\n                "models": ";".join(paths[modality]),\n                "figure": path.relative_to(destination).as_posix(),\n            })\n        sheets[(field, label)].append((rank, index, row))\n    for (field, label), items in sheets.items():\n        safe_label = "".join(c if c.isalnum() or c in "_-" else "_" for c in label)\n        source_items = [(rank, datasets[modalities[0]][index], row)\n                        for rank, index, row in items]\n        _write_contact_sheet(source_items, destination / field / safe_label / "contact_sheet.png",\n                             f"{field}: {label}", display_max, dpi)\n    with (destination / "selection.csv").open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(inventory[0]))\n        writer.writeheader()\n        writer.writerows(inventory)\n    coverage = [{"group_by": field, "group": label, "test_pairs": count,\n                 "complete_prediction_pairs": sum(\n                     1 for item in rows if (item.get(field, "unknown") or "unknown") == label\n                     and _complete_prediction_paths(item["pair_id"], evaluations) is not None)}\n                for (field, label), count in sorted(expected.items())]\n    with (destination / "coverage.csv").open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(coverage[0]))\n        writer.writeheader()\n        writer.writerows(coverage)\n    metadata = {\n        "dataset_manifest_sha256": manifest_sha,\n        "manifest_verified": not mismatches,\n        "split": split, "radiometry": radiometry, "seed": seed,\n        "selection_seed": seed,\n        "selection_method": "SHA256(seed:pair_id) order; prefer distinct spatial blocks and states",\n        "group_by": list(group_by), "per_group": per_group,\n        "eligible_pairs_with_all_model_predictions": eligible_count,\n        "models": {modality: list(models) for modality, models in evaluations.items()},\n        "border_crop_for_evaluation": border_crop,\n        "display_max": display_max, "error_max": error_max,\n        "note": "Figures use model predictions saved by locked evaluation; source display scaling does not affect metrics.",\n    }\n    (destination / "gallery_protocol.json").write_text(\n        json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")\n    (destination / "README.md").write_text(\n        "# Mahimandal qualitative candidates\\n\\n"\n        "Open `contact_sheet.png` in each group, then inspect the three detailed PNGs "\n        "under `rgb/` and `rgbnir/`. Each detailed panel includes every model with a "\n        "matching saved prediction. The first row is the full scene, second a fixed "\n        "center zoom, third RGB absolute error against Sentinel-2. All columns share "\n        "the same display and error scales. Invalid pixels are transparent in error maps.\\n\\n"\n        "Selection uses a fixed seed and favors distinct spatial blocks and states; it "\n        "does not rank learned models by test results. Candidate choices for the paper "\n        "must be documented. `selection.csv` records exact pair IDs and figure paths. "\n        "No TIFF predictions or model checkpoints are copied into this gallery.\\n",\n        encoding="utf-8")\n    archive.parent.mkdir(parents=True, exist_ok=True)\n    if archive.exists():\n        raise FileExistsError(f"Archive already exists; choose a new path: {archive}")\n    with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED, compresslevel=3) as bundle:\n        hashes = []\n        for path in sorted(destination.rglob("*")):\n            if path.is_file():\n                relative = path.relative_to(destination).as_posix()\n                bundle.write(path, relative)\n                hashes.append(f"{digest(path)}  {relative}")\n        bundle.writestr("checksums.sha256", "\\n".join(hashes) + "\\n")\n    return archive, metadata\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--dataset", type=Path, required=True)\n    parser.add_argument("--destination", type=Path, required=True)\n    parser.add_argument("--archive", type=Path, required=True)\n    parser.add_argument("--group-by", nargs="+", default=["landscape_class", "geographic_stratum"],\n                        choices=["landscape_class", "geographic_stratum", "state_or_ut"])\n    parser.add_argument("--per-group", type=int, default=3)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--split", choices=["val", "test"], default="test")\n    parser.add_argument("--modalities", nargs="+", default=["rgb", "rgbnir"],\n                        choices=["rgb", "rgbnir"])\n    parser.add_argument("--dpi", type=int, default=110)\n    args = parser.parse_args()\n    if args.per_group < 1 or args.dpi < 50:\n        parser.error("per-group must be positive and dpi must be at least 50")\n    archive, metadata = build_gallery(\n        args.output, args.dataset, args.destination, args.archive,\n        args.group_by, per_group=args.per_group, seed=args.seed,\n        split=args.split, modalities=args.modalities, dpi=args.dpi)\n    print(json.dumps(metadata, indent=2, ensure_ascii=False))\n    print("Gallery archive:", archive)\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, content in HARNESS.items():
    destination = assert_owned(HARNESS_DIR / name)
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    temporary.write_text(content, encoding='utf-8')
    temporary.replace(destination)
REGISTRY = json.loads(HARNESS['registry.json'])
print('Harness:', HARNESS_DIR)
print('Registered models:', ', '.join(REGISTRY))


## 4. Select the experiment matrix


In [ ]:
# Review replication: same frozen protocol, new independent seeds.
EPOCHS = 50
BATCH_SIZE = 1
EARLY_STOPPING_PATIENCE = 3
TRAIN_LR_CROP = 160
LEARNING_RATE = 1e-4
WORKERS = 2
CORE_MODELS = ["EDSR", "RCAN", "RDN", "CAFRN", "DCM", "FENet", "RealESRGAN_SRVGG_X3"]
EARLIER_PAPER_MODELS = []
MODELS_TO_RUN = CORE_MODELS
EXPERIMENTS = [("reflectance16", "rgb"), ("reflectance16", "rgbnir")]
SEEDS = [43, 44]
# SEEDS = [43]

RUN_TEST = False

unknown = set(MODELS_TO_RUN).difference(REGISTRY)
if unknown:
    raise ValueError(f"Unknown models: {sorted(unknown)}")
print("Models:", MODELS_TO_RUN)
print("Experiments:", EXPERIMENTS)
print("Epochs / batch / patience:", EPOCHS, BATCH_SIZE, EARLY_STOPPING_PATIENCE)
print("Test locked:", not RUN_TEST)


## 5. Audit the locked release without changing its splits


In [ ]:
run([
    PYTHON, HARNESS_DIR / "run.py", "audit",
    "--dataset", DATASET_ROOT,
    "--output", OUTPUT,
])
AUDIT = json.loads((OUTPUT / "dataset_audit.json").read_text(encoding="utf-8"))
from IPython.display import display
display(AUDIT)
if AUDIT["pairs"] != 4800:
    raise RuntimeError(f"Expected 4,800 release pairs, found {AUDIT['pairs']}")
if AUDIT["cross_split_spatial_blocks"] != 0:
    raise RuntimeError("Spatial leakage detected")
print("Locked realized split counts:", AUDIT["counts"])
print("Locked realized split fractions:", AUDIT["realized_fractions"])


## 6. Fetch pinned official model sources into 25MCSA19


In [ ]:
# adapters.fetch writes only below the explicit --sources directory.
run([PYTHON, HARNESS_DIR / "run.py", "fetch", "--sources", SOURCES])
required_source_dirs = sorted({REGISTRY[model]["source_dir"] for model in MODELS_TO_RUN})
missing_source_dirs = [name for name in required_source_dirs
                       if not (SOURCES / name / ".git").is_dir()]
if missing_source_dirs:
    raise RuntimeError(f"Official source fetch is incomplete: {missing_source_dirs}")
for model, specification in REGISTRY.items():
    source_root = assert_owned(SOURCES / specification["source_dir"], allow_missing=False)
    if source_root.is_symlink():
        raise RuntimeError(f"Model source must not be a symlink: {source_root}")


## 7. Architecture and memory preflight

Every selected model/modality executes a real 160→480 forward/backward pass.
Unsupported RGB+NIR constructors are recorded and skipped. Architectures are
not reduced after an out-of-memory error.


In [ ]:
PREFLIGHT = {}
required_source_dirs = sorted({REGISTRY[model]["source_dir"] for model in MODELS_TO_RUN})
missing_source_dirs = [name for name in required_source_dirs
                       if not (SOURCES / name / ".git").is_dir()]
if missing_source_dirs:
    raise RuntimeError(
        "Official model sources are missing: "
        f"{missing_source_dirs}. Run the preceding source-fetch cell, "
        "wait for it to finish, then rerun preflight."
    )
for radiometry, modality in EXPERIMENTS:
    for model in MODELS_TO_RUN:
        key = f"{modality}/{model}"
        if modality not in REGISTRY[model]["modalities"]:
            PREFLIGHT[key] = {
                "status": "unsupported",
                "reason": f"verified modalities={REGISTRY[model]['modalities']}",
            }
            continue
        command = [
            PYTHON, HARNESS_DIR / "run.py", "smoke",
            "--model", model, "--modality", modality,
            "--sources", SOURCES, "--output", OUTPUT,
            "--device", "cuda", "--backward",
        ]
        completed = run(command, check=False)
        PREFLIGHT[key] = {
            "status": "passed" if completed.returncode == 0 else "failed",
            "returncode": completed.returncode,
        }
preflight_path = OUTPUT / "preflight_status.json"
preflight_path.write_text(json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
display(PREFLIGHT)

def preflight_passed(modality, model):
    return PREFLIGHT.get(f"{modality}/{model}", {}).get("status") == "passed"

def train_model(model):
    if model not in MODELS_TO_RUN:
        print("Not selected:", model)
        return
    for radiometry, modality in EXPERIMENTS:
        if not preflight_passed(modality, model):
            print("Skipped failed/unsupported preflight:", modality, model)
            continue
        for seed in SEEDS:
            run([
                PYTHON, HARNESS_DIR / "run.py", "train",
                "--model", model,
                "--dataset", DATASET_ROOT,
                "--sources", SOURCES,
                "--output", OUTPUT,
                "--radiometry", radiometry,
                "--modality", modality,
                "--epochs", EPOCHS,
                "--batch-size", BATCH_SIZE,
                "--patience", EARLY_STOPPING_PATIENCE,
                "--train-lr-crop", TRAIN_LR_CROP,
                "--lr", LEARNING_RATE,
                "--workers", WORKERS,
                "--seed", seed,
                "--device", "cuda",
            ])


## Train CAFRN

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('CAFRN')


In [ ]:
train_model('CAFRN')


## Train DCM

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('DCM')


In [ ]:
train_model('DCM')

## Train FENet

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('FENet')


In [ ]:
train_model('FENet')


## Train HAUNet

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('HAUNet')


## Train HSENet

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('HSENet')


## Train MHAN

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('MHAN')


## Train SAN

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('SAN')


## Train EDSR

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('EDSR')


In [ ]:
train_model('EDSR')


## Train RCAN

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('RCAN')


## Train RDN

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('RDN')


In [ ]:
train_model('RDN')


## Train SwinIR

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('SwinIR')


## Train HAT

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('HAT')


## Train OmniSR

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('OmniSR')


## Train RealESRGAN_SRVGG_X3

Rerunning this cell resumes its last completed epoch. Best-checkpoint
selection uses full-frame validation PSNR only.


In [ ]:
train_model('RealESRGAN_SRVGG_X3')


In [ ]:
train_model('RealESRGAN_SRVGG_X3')


## 8. Validation evaluation and learning curves


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def locked_training_cli(radiometry, modality, model, seed):
    # Reuse the exact training fields embedded in this checkpoint's signature.
    if model == "bicubic":
        return []
    protocol_path = (
        OUTPUT / radiometry / modality / model / f"seed_{seed}" / "protocol.json"
    )
    if not protocol_path.is_file():
        raise FileNotFoundError(
            f"Missing checkpoint protocol: {protocol_path}. Train {model} first."
        )
    protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
    training = protocol["training"]
    return [
        "--batch-size", training["batch_size"],
        "--train-lr-crop", training["train_lr_crop"],
        "--lr", training["learning_rate"],
    ]

for radiometry, modality in EXPERIMENTS:
    for seed in SEEDS:
        for model in ["bicubic"] + MODELS_TO_RUN:
            if model != "bicubic" and not preflight_passed(modality, model):
                continue
            command = [
                PYTHON, HARNESS_DIR / "run.py", "evaluate",
                "--model", model,
                "--dataset", DATASET_ROOT,
                "--sources", SOURCES,
                "--output", OUTPUT,
                "--radiometry", radiometry,
                "--modality", modality,
                "--seed", seed,
                "--split", "val",
                "--border-crop", 3,
                "--device", "cuda",
            ]
            command.extend(locked_training_cli(radiometry, modality, model, seed))
            run(command)

histories = []
for path in OUTPUT.glob("*/*/*/seed_*/training.csv"):
    frame = pd.read_csv(path)
    parts = path.relative_to(OUTPUT).parts
    frame["radiometry"], frame["modality"], frame["model"], frame["seed"] = parts[:4]
    histories.append(frame)
if histories:
    history = pd.concat(histories, ignore_index=True)
    for (radiometry, modality), frame in history.groupby(["radiometry", "modality"]):
        figure, axis = plt.subplots(figsize=(11, 6))
        for (model, seed_label), values in frame.groupby(["model", "seed"]):
            axis.plot(values.epoch, values.val_psnr, marker="o", label=f"{model}/{seed_label}")
        axis.set(
            title=f"Validation PSNR: {radiometry} / {modality}",
            xlabel="Epoch", ylabel="PSNR (dB)",
        )
        axis.grid(alpha=0.25)
        axis.legend(ncol=2)
        plt.show()


## 9. Lock model selection, then evaluate the official test split

Leave `RUN_TEST=False` until model list, epoch ceilings and all choices are
frozen from validation only. Test evaluation always uses `best.pt` and every
official test pair; it never selects the best test run.


In [ ]:
if not RUN_TEST:
    raise RuntimeError("Set RUN_TEST=True after validation choices are fixed.")
if not RUN_TEST:
    print("Test remains locked. Set RUN_TEST=True in the controls cell only after validation decisions are final.")
else:
    for radiometry, modality in EXPERIMENTS:
        for seed in SEEDS:
            for model in ["bicubic"] + MODELS_TO_RUN:
                if model != "bicubic" and not preflight_passed(modality, model):
                    continue
                command = [
                    PYTHON, HARNESS_DIR / "run.py", "evaluate",
                    "--model", model,
                    "--dataset", DATASET_ROOT,
                    "--sources", SOURCES,
                    "--output", OUTPUT,
                    "--radiometry", radiometry,
                    "--modality", modality,
                    "--seed", seed,
                    "--split", "test",
                    "--border-crop", 3,
                    "--device", "cuda",
                ]
                command.extend(locked_training_cli(radiometry, modality, model, seed))
                run(command)


## 10. Paper tables, confidence intervals and ablations


In [ ]:
if str(HARNESS_DIR) not in sys.path:
    sys.path.insert(0, str(HARNESS_DIR))
from report import summarize, visualize, bundle

SPLIT = "test" if RUN_TEST else "val"
SUMMARY = summarize(OUTPUT, SPLIT)
display(SUMMARY.sort_values(
    ["radiometry", "modality", "psnr"], ascending=[True, True, False]
))
REPORT = OUTPUT / "reports" / SPLIT
for name in (
    "paired_vs_bicubic.csv", "by_landscape_class.csv",
    "by_geographic_stratum.csv", "rgb_vs_rgbnir.csv",
#     "reflectance16_vs_quantized8.csv",
    "complexity.csv",
):
    path = REPORT / name
    if path.is_file():
        print("\n", name)
        display(pd.read_csv(path))

metrics_figure = REPORT / "main_metrics.png"
if metrics_figure.is_file():
    from IPython.display import Image
    display(Image(filename=str(metrics_figure)))


## 11. Diverse qualitative examples using one shared display stretch


In [ ]:
from data import Pairs
from IPython.display import Image

def show_diverse_cases(
    radiometry="reflectance16", modality="rgb", seed=43,
    classes=("urban", "cropland", "forest", "water_wetland", "alpine"),
):
    dataset = Pairs(DATASET_ROOT, SPLIT, modality, radiometry)
    shown = []
    for scene_class in classes:
        index = next((
            i for i, row in enumerate(dataset.rows)
            if row.get("landscape_class") == scene_class
        ), None)
        if index is None:
            print("No case in split:", scene_class)
            continue
        path = visualize(
            OUTPUT, DATASET_ROOT, index=index,
            radiometry=radiometry, modality=modality, seed=seed,
            split=SPLIT, models=["bicubic"] + MODELS_TO_RUN,
            display_max=0.3,
        )
        display(Image(filename=str(path)))
        shown.append(path)
    return shown

QUALITATIVE = show_diverse_cases()


## 11a. Three complete-model examples per landscape and geographic stratum

Run after the locked **test** evaluation finishes for both RGB and RGB+NIR.
This cell reads saved full-frame predictions and does not run inference.
It selects three distinct test pairs per landscape class and per broad
geographic stratum, preferring distinct spatial blocks/states. Every
displayed benchmark model must have a saved prediction for each pair.
The figures, contact sheets, selection CSV and protocol are packaged in
one ZIP under the owned `25mcsa19` archive directory.


In [ ]:
# from qualitative_gallery import build_gallery
# from IPython.display import FileLink

# GALLERY_STAMP = time.strftime("%Y%m%dT%H%M%S")
# GALLERY_DIR = assert_owned(ARCHIVE_ROOT / f"qualitative_test_3_per_group_{GALLERY_STAMP}")
# GALLERY_ZIP = assert_owned(ARCHIVE_ROOT / f"qualitative_test_3_per_group_{GALLERY_STAMP}.zip")
# gallery_zip, gallery_protocol = build_gallery(
#     OUTPUT, DATASET_ROOT, GALLERY_DIR, GALLERY_ZIP,
#     group_by=("landscape_class", "geographic_stratum"),
#     per_group=3, seed=42, split="test", radiometry="reflectance16",
#     modalities=("rgb", "rgbnir"), border_crop=3,
#     display_max=0.3, error_max=0.10, dpi=110,
# )
# display(pd.read_csv(GALLERY_DIR / "selection.csv"))
# display(pd.read_csv(GALLERY_DIR / "coverage.csv"))
# display(FileLink(str(gallery_zip)))
import time
import pandas as pd
from qualitative_gallery import build_gallery
from IPython.display import FileLink, display

GALLERY_SEED = 43  # Use 44 for the other completed seed.
GALLERY_STAMP = time.strftime("%Y%m%dT%H%M%S")

GALLERY_DIR = assert_owned(
    ARCHIVE_ROOT /
    f"qualitative_test_seed{GALLERY_SEED}_{GALLERY_STAMP}"
)
GALLERY_ZIP = assert_owned(
    ARCHIVE_ROOT /
    f"qualitative_test_seed{GALLERY_SEED}_{GALLERY_STAMP}.zip"
)

gallery_zip, gallery_protocol = build_gallery(
    OUTPUT, DATASET_ROOT, GALLERY_DIR, GALLERY_ZIP,
    group_by=("landscape_class", "geographic_stratum"),
    per_group=3,
    seed=GALLERY_SEED,
    split="test",
    radiometry="reflectance16",
    modalities=("rgb", "rgbnir"),
    border_crop=3,
    display_max=0.3,
    error_max=0.10,
    dpi=110,
)

display(pd.read_csv(GALLERY_DIR / "selection.csv"))
display(pd.read_csv(GALLERY_DIR / "coverage.csv"))
display(FileLink(str(gallery_zip)))

## 12. Package reproducible evidence inside 25MCSA19

The paper bundle contains protocols, source revisions, environments,
learning curves, aggregate/per-image tables, confidence intervals and
figures. Dataset files are never copied into result archives or deleted.


In [ ]:
from pathlib import Path
import zipfile
import stat

# Locate only your own roll-number folder.
roll_root = next(
    p for p in (Path(WORKSPACE_ROOT).resolve(),)
    + tuple(Path(WORKSPACE_ROOT).resolve().parents)
    if p.name.casefold() == "25mcsa19"
)

# Change this filename if your uploaded ZIP has a different name.
archive = roll_root / (
    "mahimandal160_paper_benchmark_v1_complete_compact_"
    "20261002T113656696649Z.zip"
)

restore = assert_owned(WORKSPACE_ROOT / "restored_seed42")
restore.mkdir(parents=True, exist_ok=True)

def find_original():
    return sorted({
        p.parent
        for p in restore.rglob("preflight_status.json")
        if (p.parent / "reflectance16").is_dir()
    })

matches = find_original()

if not matches:
    if not archive.is_file():
        raise FileNotFoundError(
            f"Set archive to your uploaded benchmark-results ZIP: {archive}"
        )

    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (restore / member.filename).resolve()
            target.relative_to(restore.resolve())

            if stat.S_ISLNK(member.external_attr >> 16):
                raise RuntimeError(f"Unsafe symlink: {member.filename}")
            if target.exists() and not member.is_dir():
                raise FileExistsError(
                    f"Partial extraction exists: {target}. "
                    "Choose a new restore folder; nothing was overwritten."
                )

        z.extractall(restore)

    matches = find_original()

if len(matches) != 1:
    raise RuntimeError(f"Expected one original benchmark folder: {matches}")

ORIGINAL_RESULTS = assert_owned(matches[0], allow_missing=False)
print("Original seed-42 results:", ORIGINAL_RESULTS)

In [ ]:
# Combine original seed 42 with new seeds 43/44 only under matching protocols.
import copy
import pandas as pd

def without_seed(protocol):
    value = copy.deepcopy(protocol)
    value["training"].pop("seed")
    return value

if not RUN_TEST:
    print("Three-seed test summary remains locked.")
else:
#     original = assert_owned(WORKSPACE_ROOT / "outputs" / "mahimandal160_paper_benchmark_v1", allow_missing=False)
    original = ORIGINAL_RESULTS
    rows = []
    contracts = {}
    for root, accepted_seeds in ((original, {42}), (OUTPUT, {43, 44})):
        for path in root.glob("reflectance16/*/*/seed_*/evaluation/test/*/metrics.json"):
            parts = path.relative_to(root).parts
            modality, model, seed = parts[1], parts[2], int(parts[3][5:])
            if model == "bicubic" or model not in MODELS_TO_RUN or seed not in accepted_seeds:
                continue
            run_root = root / "reflectance16" / modality / model / ("seed_" + str(seed))
            protocol = json.loads((run_root / "protocol.json").read_text())
            contract = without_seed(protocol)
            key = modality, model
            if key in contracts and contracts[key] != contract:
                raise RuntimeError("Different source/model/data/training protocol: " + str(key))
            contracts[key] = contract
            metric = json.loads(path.read_text())
            if metric.get("count") != 731 or metric.get("border_crop") != 3:
                raise RuntimeError("Unexpected test cohort or evaluation border")
            rows.append(dict(modality=modality, model=model, seed=seed,
                             psnr=metric["psnr"], ssim=metric["ssim"], mae=metric["mae"]))
    frame = pd.DataFrame(rows)
    if frame.empty or frame.duplicated(["modality", "model", "seed"]).any():
        raise RuntimeError("Missing or duplicate seed results")
    seed_sets = frame.groupby(["modality", "model"]).seed.agg(lambda values: sorted(values))
    print("Available seeds per model; no best-seed selection:")
    display(seed_sets)
    complete = [key for key, seeds in seed_sets.items() if seeds == [42, 43, 44]]
    selected = frame.set_index(["modality", "model"]).loc[complete].reset_index() if complete else frame.iloc[:0]
    destination = assert_owned(OUTPUT / "reports" / "three_seed_review")
    destination.mkdir(parents=True, exist_ok=True)
    frame.to_csv(destination / "per_seed.csv", index=False)
    if len(selected):
        summary = selected.groupby(["modality", "model"]).agg(
            seeds=("seed", "nunique"), psnr_mean=("psnr", "mean"), psnr_std=("psnr", "std"),
            ssim_mean=("ssim", "mean"), ssim_std=("ssim", "std"), mae_mean=("mae", "mean"))
        summary.to_csv(destination / "three_seed_mean_std.csv")
        display(summary)
    print("Seed SD is fit variability; it does not replace the 13-block geographic analysis.")


In [ ]:
from IPython.display import FileLink

paper_archive = Path(bundle(OUTPUT, include_predictions=False))
assert_owned(paper_archive, allow_missing=False)
display(FileLink(str(paper_archive)))


In [ ]:
print("Full-test prediction archive disabled; use the compact checkpoint export below.")


In [ ]:
print("Unfiltered restart archive disabled; compact export below keeps checkpoints without full-test rasters.")


In [ ]:
# Download checkpoint-bearing results without full-test prediction rasters.
from datetime import datetime, timezone
import zipfile
from IPython.display import FileLink, display

if not RUN_TEST:
    raise RuntimeError("Complete the locked test stage and three-seed summary before final export.")
root = assert_owned(OUTPUT, allow_missing=False)
archive = assert_owned(ARCHIVE_ROOT / ("mahimandal_review_seeds_complete_compact_" +
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".zip"))
inventory = []
checkpoint_count = 0
with zipfile.ZipFile(archive, "x", allowZip64=True) as output_zip:
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.is_symlink():
            continue
        assert_owned(path, allow_missing=False)
        relative = path.relative_to(root)
        if "predictions" in relative.parts or path.suffix.lower() in {".zip", ".tif", ".tiff", ".npy", ".npz"}:
            continue
        member = "results/" + relative.as_posix()
        output_zip.write(path, member, compress_type=(zipfile.ZIP_STORED if path.suffix.lower() == ".pt" else zipfile.ZIP_DEFLATED))
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
        inventory.append(digest.hexdigest() + "  " + member)
        checkpoint_count += path.suffix.lower() == ".pt"
    for path in sorted(HARNESS_DIR.iterdir()):
        if path.is_file() and path.suffix in {".py", ".json"}:
            assert_owned(path, allow_missing=False)
            member = "code/" + path.name
            output_zip.write(path, member, compress_type=zipfile.ZIP_DEFLATED)
            inventory.append(hashlib.sha256(path.read_bytes()).hexdigest() + "  " + member)
    notebook_path = SESSION_ROOT / "Mahimandal_Review_Additional_Seeds_DGX_25MCSA19.ipynb"
    if notebook_path.is_file():
        assert_session_path(notebook_path, allow_missing=False)
        member = "executed_notebook/" + notebook_path.name
        output_zip.write(notebook_path, member, compress_type=zipfile.ZIP_DEFLATED)
        inventory.append(hashlib.sha256(notebook_path.read_bytes()).hexdigest() + "  " + member)
    output_zip.writestr("checksums.sha256", "\n".join(inventory) + "\n")
print("Checkpoint files:", checkpoint_count)
print("Full-test prediction rasters excluded. Download:", archive)
display(FileLink(str(archive)))
